# Prompt engineering with Llama

Emotion classification of Bulgarian sentences with Llama models, from a zero-shot baseline to self-consistency voting. The LLM server address is read from the environment variables `OPENWEBUI_URL` and `OPENWEBUI_API_KEY`.

In [1]:
import os
import requests
import json
import time
from sklearn.metrics import f1_score, classification_report
import numpy as np

In [2]:
print(f"NumPy version: {np.__version__}")

NumPy version: 1.26.4


In [3]:
import pandas as pd
df = pd.read_csv("Test_set.csv")
df = df.rename(columns={"Emotion_core": "Label"})
df = df[["Sentence", "Label"]]

In [23]:
df.to_csv("Prompt_data.csv", index=False)

In [5]:
df.head()

,Sentence,Label
0,"Ако приемаш моите извинения, просто ме пригърни.",sadness
1,"Трябвам това, че си ме пригърни.",sadness
2,С теб всеки миски е прекрасен. Обичам.,happiness
3,Аз наистина ги гледам по страни и се вълнувам ...,happiness
4,"Ти избра играта, а аз избрах любовта.",sadness


In [6]:
df.shape

(1005, 2)

In [7]:
os.environ["OPENWEBUI_API_KEY"] = "sk-463b5b0d1e1041b396945c0c6115e220"
df.shape

(1005, 2)

In [8]:
API_BASE = os.environ["OPENWEBUI_URL"]

## Verify API Access and List Models

In [9]:
import requests
import json
import os

# Get API token
API_TOKEN = os.environ.get("OPENWEBUI_API_KEY")
API_BASE = os.environ["OPENWEBUI_URL"]
API_URL = f"{API_BASE}/api/chat/completions"

# Headers
header = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}

# Test connection
response = requests.get(f"{API_BASE}/api/models", headers=header)

# Check status code first
print(f"Status Code: {response.status_code}")
print(f"\nRaw Response Text:")
print(response.text[:500])  # First 500 characters

# Only try to parse JSON if status is 200
if response.status_code == 200:
    try:
        print(f"\nJSON Response:")
        print(json.dumps(response.json(), indent=2))
    except:
        print("\nCouldn't parse as JSON. Raw content:")
        print(response.text)
else:
    print(f"\nError: Server returned status {response.status_code}")
    print(response.text)

Status Code: 200

Raw Response Text:
{"data":[{"id":"llama3.2:3b","object":"model","created":1761575762,"owned_by":"openai","root":"meta-llama/Llama-3.2-3B-Instruct","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-94129ba5150345bda4abb2b292c1d9d2","object":"model_permission","created":1761575762,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}],"connection_type":"exte

JSON Response:
{
  "data": [
    {
      "id": "llama3.2:3b",
      "object": "model",
      "created": 1761575762,
      "owned_by": "openai",
      "root": "meta-llama/Llama-3.2-3B-Instruct",
      "parent": null,
      "max_model_len": 8192,
      "permission": [
        {
          "id": "modelperm-94129ba5150345bda4abb2b292c1d9d2",
          "object": "model_permission",
          "created": 1761575762,
          "allow_create_engine": false,
         

# LLama 3.2:3B models

## Iteration 1: Baseline Emotion Classification Function

In [10]:
headers = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}

# Function to classify one sentence with BASELINE PROMPT
def classify_emotion(sentence):
    """
    Baseline prompt: Simple classification request
    """
    data = {
        "model": "llama3.2:3b",
        "messages": [{
            "role": "user",
            "content": f"""Analyze the following sentence and classify it as one of the six core emotions (happiness, sadness, anger, surprise, fear, disgust) or neutral: {sentence}

Respond with only the emotion label."""
        }],
        "temperature": 0.1,
        "max_tokens": 20
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=30)
        if response.status_code == 200:
            result = response.json()
            predicted = result['choices'][0]['message']['content'].strip().lower()
            return predicted
        else:
            return "error"
    except Exception as e:
        print(f"Error: {e}")
        return "error"

In [11]:
sample_size = 42
sample_df = df.head(sample_size).copy()

predictions = []
print(f"\nClassifying {sample_size} sentences...\n")

for idx, row in sample_df.iterrows():
    sentence = row['Sentence']
    true_label = row['Label']

    print(f"[{idx+1}/{sample_size}] Classifying...")
    print(f"  Text: {sentence[:60]}...")

    # Get prediction
    predicted = classify_emotion(sentence)
    predictions.append(predicted)

    # Show result
    match = "✓" if predicted.lower() == true_label.lower() else "✗"
    print(f"  True: {true_label:12} | Predicted: {predicted:12} {match}\n")

    # Rate limiting - wait between requests
    time.sleep(0.5)


Classifying 42 sentences...

[1/42] Classifying...
  Text: Ако приемаш моите извинения, просто ме пригърни....
  True: sadness      | Predicted: politeness   ✗

[2/42] Classifying...
  Text: Трябвам това, че си ме пригърни....
  True: sadness      | Predicted: disgust      ✗

[3/42] Classifying...
  Text: С теб всеки миски е прекрасен. Обичам....
  True: happiness    | Predicted: love         ✗

[4/42] Classifying...
  Text: Аз наистина ги гледам по страни и се вълнувам за тези хора....
  True: happiness    | Predicted: surprise     ✗

[5/42] Classifying...
  Text: Ти избра играта, а аз избрах любовта....
  True: sadness      | Predicted: love         ✗

[6/42] Classifying...
  Text: Не е така. По-скоро бих казал това на теб....
  True: anger        | Predicted: disgust      ✗

[7/42] Classifying...
  Text: Днес ви очаквам, че възможността си възможност,...
  True: neutral      | Predicted: neutral      ✓

[8/42] Classifying...
  Text: аз избрах това на теб....
  True: happiness    | 

In [12]:
# Add predictions to dataframe
sample_df['predicted'] = predictions

# Calculate metrics
y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted'].str.lower()

# Calculate F1 score
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

# Count matches
matches = (y_true == y_pred).sum()
accuracy = matches / len(sample_df)

print(f"\nSample Size: {sample_size}")
print(f"Correct: {matches}/{sample_size}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")
print(f"\nTarget F1: 0.85")
print(f"Gap: {0.85 - f1_macro:.4f}")


Sample Size: 42
Correct: 12/42
Accuracy: 0.2857
F1 Score (Macro): 0.1622
F1 Score (Weighted): 0.3415

Target F1: 0.85
Gap: 0.6878


In [13]:
print(classification_report(y_true, y_pred, zero_division=0))

sample_df.to_csv("baseline_results.csv", index=False)

              precision    recall  f1-score   support

       anger       0.00      0.00      0.00         2
 contentment       0.00      0.00      0.00         0
   curiosity       0.00      0.00      0.00         0
     disgust       0.08      1.00      0.15         1
        fear       1.00      0.60      0.75         5
   happiness       0.50      0.09      0.15        11
        love       0.00      0.00      0.00         0
     neutral       0.60      0.33      0.43         9
   nostalgia       0.00      0.00      0.00         0
  politeness       0.00      0.00      0.00         0
     respect       0.00      0.00      0.00         0
     sadness       0.60      0.30      0.40        10
    surprise       0.20      0.25      0.22         4

    accuracy                           0.29        42
   macro avg       0.23      0.20      0.16        42
weighted avg       0.54      0.29      0.34        42



In [14]:
# Show detailed report
print(classification_report(y_true, y_pred, zero_division=0))

# Save results
sample_df.to_csv("baseline_results_10.csv", index=False)
print("\n✓ Results saved to baseline_results_10.csv")

# Show all predictions
for idx, row in sample_df.iterrows():
    match = "✓" if row['predicted'].lower() == row['Label'].lower() else "✗"
    print(f"{match} True: {row['Label']:12} | Pred: {row['predicted']:12} | {row['Sentence'][:50]}...")

              precision    recall  f1-score   support

       anger       0.00      0.00      0.00         2
 contentment       0.00      0.00      0.00         0
   curiosity       0.00      0.00      0.00         0
     disgust       0.08      1.00      0.15         1
        fear       1.00      0.60      0.75         5
   happiness       0.50      0.09      0.15        11
        love       0.00      0.00      0.00         0
     neutral       0.60      0.33      0.43         9
   nostalgia       0.00      0.00      0.00         0
  politeness       0.00      0.00      0.00         0
     respect       0.00      0.00      0.00         0
     sadness       0.60      0.30      0.40        10
    surprise       0.20      0.25      0.22         4

    accuracy                           0.29        42
   macro avg       0.23      0.20      0.16        42
weighted avg       0.54      0.29      0.34        42


✓ Results saved to baseline_results_10.csv
✗ True: sadness      | Pred: polite

In [15]:
df["Label"].value_counts()

Label
neutral      452
sadness      188
happiness    187
fear          66
anger         51
surprise      39
disgust       22
Name: count, dtype: int64

## Function to Extract Emotion Label

In [16]:
import re
# Function to extract ONLY the emotion label from LLM response
def extract_emotion_label(response_text):
    if not response_text:
        return "neutral"

    # Clean the response
    response_text = response_text.lower().strip()

    # Valid emotions - ONLY these 7!
    valid_emotions = ["happiness", "sadness", "anger", "surprise", "fear", "disgust", "neutral"]

    # Pattern 1: "emotion: XXXX" or "Emotion: XXXX"
    emotion_pattern = r'emotion:\s*(\w+)'
    match = re.search(emotion_pattern, response_text, re.IGNORECASE)
    if match:
        extracted = match.group(1).lower()
        if extracted in valid_emotions:
            return extracted

    # Pattern 2: Check first line for emotion
    first_line = response_text.split('\n')[0].strip()
    for emotion in valid_emotions:
        if emotion in first_line:
            return emotion

    # Pattern 3: Check entire text for any valid emotion (first occurrence)
    for emotion in valid_emotions:
        if emotion in response_text:
            return emotion

    # Default: neutral if nothing found
    return "neutral"

## Iteration 2: Few-shot Prompt

In [17]:
# API setup
API_TOKEN = os.environ.get("OPENWEBUI_API_KEY")
API_URL = os.environ["OPENWEBUI_URL"] + "/api/chat/completions"

headers = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}

# Modified Few-Shot Function - NOW SHOWS FULL RESPONSE
def classify_emotion_fewshot(sentence, show_full_response=True):
    data = {
        "model": "llama3.2:3b",
        "messages": [
            {
                "role": "system",
                "content": "You are an expert emotion classifier for Bulgarian text. Classify into one of these emotions ONLY: happiness, sadness, anger, surprise, fear, disgust, neutral."
            },
            {
                "role": "user",
                "content": f"""Classify the emotion in Bulgarian sentences. Examples:

Sentence: Щастлив съм, когато съм с теб!
Emotion: happiness

Sentence: Тази ситуация ме натъжи.
Emotion: sadness

Sentence: Толкова съм ти ядосан!
Emotion: anger

Sentence: Еха, не го очаквах!
Emotion: surprise

Sentence: Страх ме е от това.
Emotion: fear

Sentence: Това е нещо отвратително.
Emotion: disgust

Sentence: Добре.
Emotion: neutral

Now classify:
Sentence: {sentence}
Emotion: """
            }
        ],
        "temperature": 0.1,
        "max_tokens": 100
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=30)
        if response.status_code == 200:
            result = response.json()

            # Get the FULL response from the model
            full_response = result['choices'][0]['message']['content']

            # Show the full response if requested
            if show_full_response:
                print(f"Full LLM Response: '{full_response}'")

            # Extract ONLY the emotion label for evaluation
            extracted_label = extract_emotion_label(full_response)

            return extracted_label, full_response
        else:
            return "error", "error"
    except Exception as e:
        print(f"Error: {e}")
        return "error", "error"

In [18]:
# Test on 42 sentences
sample_size = 42
sample_df = df.head(sample_size).copy()

predictions = []
full_responses = []

print(f"\nClassifying {sample_size} sentences...\n")

for idx, row in sample_df.iterrows():
    sentence = row['Sentence']
    true_label = row['Label']

    print(f"Input Text: {sentence}")
    print(f"True Label: {true_label}")

    # Get prediction - NOW shows full response
    predicted, full_resp = classify_emotion_fewshot(sentence, show_full_response=True)

    predictions.append(predicted)
    full_responses.append(full_resp)

    # Show cleaned prediction
    print(f"Extracted Emotion: {predicted}")

    # Show if match
    match = "✓" if predicted.lower() == true_label.lower() else "✗"
    print(f"Result: {match} {'CORRECT' if match == '✓' else 'INCORRECT'}")

    # Rate limiting
    time.sleep(0.5)


Classifying 42 sentences...

Input Text: Ако приемаш моите извинения, просто ме пригърни.
True Label: sadness
Full LLM Response: 'Emotion: neutral'
Extracted Emotion: neutral
Result: ✗ INCORRECT
Input Text: Трябвам това, че си ме пригърни.
True Label: sadness
Full LLM Response: 'Emotion: happiness'
Extracted Emotion: happiness
Result: ✗ INCORRECT
Input Text: С теб всеки миски е прекрасен. Обичам.
True Label: happiness
Full LLM Response: 'Emotion: happiness'
Extracted Emotion: happiness
Result: ✓ CORRECT
Input Text: Аз наистина ги гледам по страни и се вълнувам за тези хора.
True Label: happiness
Full LLM Response: 'Emotion: happiness'
Extracted Emotion: happiness
Result: ✓ CORRECT
Input Text: Ти избра играта, а аз избрах любовта.
True Label: sadness
Full LLM Response: 'Emotion: sadness'
Extracted Emotion: sadness
Result: ✓ CORRECT
Input Text: Не е така. По-скоро бих казал това на теб.
True Label: anger
Full LLM Response: 'Emotion: sadness'
Extracted Emotion: sadness
Result: ✗ INCORREC

In [19]:
# Add to dataframe
sample_df['predicted_fewshot'] = predictions
sample_df['full_llm_response'] = full_responses

# Calculate metrics

y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted_fewshot'].str.lower()

f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
matches = (y_true == y_pred).sum()
accuracy = matches / len(sample_df)

print(f"\nSample Size: {sample_size}")
print(f"Correct: {matches}/{sample_size}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")


Sample Size: 42
Correct: 24/42
Accuracy: 0.5714
F1 Score (Macro): 0.4913
F1 Score (Weighted): 0.5725


In [20]:
# Show detailed report
print("CLASSIFICATION REPORT")
print(classification_report(y_true, y_pred, zero_division=0))

# Save with full responses
sample_df.to_csv("fewshot_results_with_full_responses.csv", index=False)
print("\n✓ Results saved to fewshot_results_with_full_responses.csv")
print("  (CSV includes full LLM responses)")

# Show summary table
print(f"{'#':<3} {'True':<12} {'Predicted':<12} {'Full Response':<30} {'Match'}")
for idx, row in sample_df.iterrows():
    match = "✓" if row['predicted_fewshot'].lower() == row['Label'].lower() else "✗"
    full_resp_short = row['full_llm_response'][:27] + "..." if len(row['full_llm_response']) > 30 else row['full_llm_response']
    print(f"{idx+1:<3} {row['Label']:<12} {row['predicted_fewshot']:<12} {full_resp_short:<30} {match}")

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger       0.00      0.00      0.00         2
     disgust       0.50      1.00      0.67         1
        fear       1.00      0.80      0.89         5
   happiness       0.67      0.73      0.70        11
     neutral       0.55      0.67      0.60         9
     sadness       0.71      0.50      0.59        10
    surprise       0.00      0.00      0.00         4

    accuracy                           0.57        42
   macro avg       0.49      0.53      0.49        42
weighted avg       0.59      0.57      0.57        42


✓ Results saved to fewshot_results_with_full_responses.csv
  (CSV includes full LLM responses)
#   True         Predicted    Full Response                  Match
1   sadness      neutral      Emotion: neutral               ✗
2   sadness      happiness    Emotion: happiness             ✗
3   happiness    happiness    Emotion: happiness             ✓
4   happiness    happiness   

## Iteration 3: Structured instructions approach

In [21]:
# API Configuration (if not already set in this session)
API_TOKEN = os.environ.get("OPENWEBUI_API_KEY")
API_URL = os.environ["OPENWEBUI_URL"] + "/api/chat/completions"

headers = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}

def classify_emotion_structured(sentence, show_full_response=True):
    """
    Classify emotion using structured prompt with explicit definitions.

    Args:
        sentence: Input text to classify
        show_full_response: Whether to print full model response

    Returns:
        tuple: (extracted_label, full_response)
    """
    data = {
        "model": "llama3.2:3b",
        "messages": [
            {
                "role": "system",
                "content": """You are an expert emotion classifier for Bulgarian text.

EMOTION DEFINITIONS (based on Plutchik's emotion wheel):
- happiness: joy at center, ranging from serenity to ecstasy. Includes: optimistic, cheerful, proud, content, peaceful, joyful
- sadness: sorrow at center, ranging from pensiveness to grief. Includes: lonely, disappointed, hurt, depressed, abandoned
- anger: rage at center, ranging from annoyance to fury. Includes: frustrated, hostile, bitter, resentful, violated, furious
- surprise: amazement at center, ranging from distraction to astonishment. Includes: shocked, confused, stunned, startled
- fear: terror at center, ranging from apprehension to panic. Includes: scared, anxious, worried, insecure, threatened, nervous
- disgust: loathing at center, ranging from boredom to hatred. Includes: disapproving, revolted, awful, nauseated, repelled
- neutral: calm, balanced state with no strong emotion. Includes: peaceful, indifferent, composed, relaxed
"""
            },
            {
                "role": "user",
                "content": f"""Classify the emotion in this Bulgarian sentence using the definitions provided.

Examples:

Sentence: Толкова съм щастлив днес!
Emotion: happiness

Sentence: Това ме разстройва много.
Emotion: sadness

Sentence: Ядосан съм на тази ситуация!
Emotion: anger

Sentence: Wow, не очаквах това!
Emotion: surprise

Sentence: Страх ме е от това.
Emotion: fear

Sentence: Това е отвратително.
Emotion: disgust

Sentence: Добре.
Emotion: neutral

Now classify:
Sentence: {sentence}
Emotion: """
            }
        ],
        "temperature": 0.1,
        "max_tokens": 100
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=30)
        if response.status_code == 200:
            result = response.json()
            full_response = result['choices'][0]['message']['content']

            if show_full_response:
                print(f"  Full LLM Response: '{full_response}'")

            extracted_label = extract_emotion_label(full_response)
            return extracted_label, full_response
        else:
            return "error", "error"
    except Exception as e:
        print(f"Error: {e}")
        return "error", "error"


In [22]:
# Execute Experiment 3: Structured Prompt
sample_size = 42
sample_df = df.head(sample_size).copy()

predictions = []
full_responses = []

for idx, row in sample_df.iterrows():
    sentence = row['Sentence']
    true_label = row['Label']

    print(f"\n[{idx+1}/{sample_size}]")
    print(f"Input: {sentence[:60]}...")
    print(f"True Label: {true_label}")

    extracted, full_resp = classify_emotion_structured(sentence, show_full_response=True)

    predictions.append(extracted)
    full_responses.append(full_resp)

    print(f"Extracted Label: {extracted}")
    match = "CORRECT" if extracted.lower() == true_label.lower() else "INCORRECT"
    print(f"Result: {match}")

    time.sleep(0.5)

sample_df['predicted_structured'] = predictions
sample_df['full_response_structured'] = full_responses


[1/42]
Input: Ако приемаш моите извинения, просто ме пригърни....
True Label: sadness
  Full LLM Response: 'Based on the definitions provided, I would classify the emotion in the sentence "Ако приемаш моите извинения, просто ме пригърни" as neutral.

The sentence is a polite and courteous request, with no strong emotional tone. It is a calm and composed statement, without any indication of happiness, sadness, anger, surprise, fear, or disgust. The tone is friendly and respectful, but it does not convey any strong emotions. Therefore, I would classify the emotion in this sentence as neutral.'
Extracted Label: neutral
Result: INCORRECT

[2/42]
Input: Трябвам това, че си ме пригърни....
True Label: sadness
  Full LLM Response: 'Based on the definition of emotions provided, I would classify the emotion in the sentence "Трябвам това, че си ме пригърни" as happiness.

The reason for this classification is that the sentence expresses a positive sentiment, with the speaker expressing gratitud

In [23]:
# Evaluation
y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted_structured'].str.lower()

f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
matches = (y_true == y_pred).sum()
accuracy = matches / len(sample_df)

print(f"Correct Predictions: {matches}/{sample_size}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")
print(f"Target F1: 0.85")
print(f"Gap to Target: {0.85 - f1_macro:.4f}")

Correct Predictions: 22/42
Accuracy: 0.5238
F1 Score (Macro): 0.4055
F1 Score (Weighted): 0.5416
Target F1: 0.85
Gap to Target: 0.4445


In [24]:
print(classification_report(y_true, y_pred, zero_division=0))

sample_df.to_csv("structured_results.csv", index=False)
print("\nResults saved to structured_results.csv")

              precision    recall  f1-score   support

       anger       0.00      0.00      0.00         2
     disgust       0.00      0.00      0.00         1
        fear       0.67      0.80      0.73         5
   happiness       0.56      0.45      0.50        11
     neutral       0.45      0.56      0.50         9
     sadness       0.88      0.70      0.78        10
    surprise       0.50      0.25      0.33         4

    accuracy                           0.52        42
   macro avg       0.44      0.39      0.41        42
weighted avg       0.58      0.52      0.54        42


Results saved to structured_results.csv


## Iteration 4: Using Multiple Techniques

In [25]:
def classify_emotion_optimized(sentence, show_full_response=True):
    """
    Optimized prompt combining:
    - Few-shot learning (most effective from results)
    - Chain-of-thought reasoning
    - Role prompting
    - Clear constraints and output formatting
    - Self-consistency through explicit rules
    """
    data = {
        "model": "llama3.2:3b",
        "messages": [
            {
                "role": "system",
                "content": """You are an expert emotion classifier specializing in Bulgarian dialogue analysis.

TASK: Classify text into exactly one of these 7 emotions:
happiness, sadness, anger, surprise, fear, disgust, neutral

CLASSIFICATION APPROACH:
1. Identify emotional keywords and phrases
2. Consider context and speaker intent
3. Determine primary emotion (if multiple present, choose dominant one)
4. Select from the 7 categories only

CONSTRAINTS:
- Use only the 7 emotions listed above
- Respond in format "Emotion: [label]"
- If ambiguous, choose neutral
- Focus on expressed emotion, not topic content"""
            },
            {
                "role": "user",
                "content": f"""Classify the emotion in the following Bulgarian sentence.

EXAMPLES:

Input: Толкова съм щастлив днес!
Analysis: Keywords "щастлив" (happy) indicate joy
Emotion: happiness

Input: Това ме разстройва много.
Analysis: "разстройва" (upsets) indicates distress
Emotion: sadness

Input: Ядосан съм на тази ситуация!
Analysis: "Ядосан" (angry) shows clear anger
Emotion: anger

Input: Wow, не очаквах това!
Analysis: "не очаквах" (didn't expect) indicates surprise
Emotion: surprise

Input: Страх ме е от това.
Analysis: "Страх" (fear) directly states fear
Emotion: fear

Input: Това е отвратително.
Analysis: "отвратително" (disgusting) shows disgust
Emotion: disgust

Input: Добре.
Analysis: Neutral acknowledgment, no emotion
Emotion: neutral

NOW CLASSIFY:

Input: {sentence}
Analysis:"""
            }
        ],
        "temperature": 0.05,
        "max_tokens": 150
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=30)
        if response.status_code == 200:
            result = response.json()
            full_response = result['choices'][0]['message']['content']

            if show_full_response:
                print(f"  Full LLM Response: '{full_response}'")

            extracted_label = extract_emotion_label(full_response)
            return extracted_label, full_response
        else:
            return "error", "error"
    except Exception as e:
        print(f"Error: {e}")
        return "error", "error"

In [ ]:
# Execute Experiment 4: Optimized Prompt
sample_size = 1007
sample_df = df.head(sample_size).copy()

predictions = []
full_responses = []

for idx, row in sample_df.iterrows():
    sentence = row['Sentence']
    true_label = row['Label']

    print(f"\n[{idx+1}/{sample_size}]")
    print(f"Input: {sentence[:60]}...")
    print(f"True Label: {true_label}")

    extracted, full_resp = classify_emotion_optimized(sentence, show_full_response=True)

    predictions.append(extracted)
    full_responses.append(full_resp)

    print(f"Extracted Label: {extracted}")
    match = "CORRECT" if extracted.lower() == true_label.lower() else "INCORRECT"
    print(f"Result: {match}")

    time.sleep(0.5)

sample_df['predicted_optimized'] = predictions
sample_df['full_response_optimized'] = full_responses


[1/1007]
Input: Ако приемаш моите извинения, просто ме пригърни....
True Label: sadness
  Full LLM Response: 'Emotion: neutral'
Extracted Label: neutral
Result: INCORRECT

[2/1007]
Input: Трябвам това, че си ме пригърни....
True Label: sadness
  Full LLM Response: 'Emotion: happiness'
Extracted Label: happiness
Result: INCORRECT

[3/1007]
Input: С теб всеки миски е прекрасен. Обичам....
True Label: happiness
  Full LLM Response: 'Emotion: happiness'
Extracted Label: happiness
Result: CORRECT

[4/1007]
Input: Аз наистина ги гледам по страни и се вълнувам за тези хора....
True Label: happiness
  Full LLM Response: 'Emotion: happiness'
Extracted Label: happiness
Result: CORRECT

[5/1007]
Input: Ти избра играта, а аз избрах любовта....
True Label: sadness
  Full LLM Response: 'Emotion: sadness'
Extracted Label: sadness
Result: CORRECT

[6/1007]
Input: Не е така. По-скоро бих казал това на теб....
True Label: anger
  Full LLM Response: 'Emotion: neutral'
Extracted Label: neutral
Result: IN

In [53]:
# Evaluation
y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted_optimized'].str.lower()

f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
matches = (y_true == y_pred).sum()
accuracy = matches / len(sample_df)

print(f"\nSample Size: {sample_size}")
print(f"Correct Predictions: {matches}/{sample_size}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")
print(f"Target F1: 0.85")
print(f"Gap to Target: {0.85 - f1_macro:.4f}")



Sample Size: 1007
Correct Predictions: 576/1007
Accuracy: 0.5731
F1 Score (Macro): 0.3615
F1 Score (Weighted): 0.5534
Target F1: 0.85
Gap to Target: 0.4885


In [54]:
# Comparison with all previous experiments
comparison_data = {
    'Baseline': 0.1360,
    'Few-Shot': 0.4917,
    'Structured': 0.4144,
    'Optimized': f1_macro
}

for name, score in comparison_data.items():
    indicator = " <- BEST" if score == max(comparison_data.values()) else ""
    print(f"{name:<15} F1 Macro: {score:.4f}{indicator}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, zero_division=0))

sample_df.to_csv("optimized_results.csv", index=False)
print("\nResults saved to optimized_results.csv")

Baseline        F1 Macro: 0.1360
Few-Shot        F1 Macro: 0.4917 <- BEST
Structured      F1 Macro: 0.4144
Optimized       F1 Macro: 0.3615

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger       0.17      0.18      0.17        51
     disgust       0.15      0.09      0.11        22
        fear       0.58      0.21      0.31        66
   happiness       0.64      0.59      0.61       187
     neutral       0.62      0.78      0.69       452
     sadness       0.54      0.44      0.48       188
    surprise       0.24      0.10      0.14        39

    accuracy                           0.57      1005
   macro avg       0.42      0.34      0.36      1005
weighted avg       0.56      0.57      0.55      1005


Results saved to optimized_results.csv


## Iteration 5: Focused Prompting on Difficult Emotions

In [55]:
# API Configuration (if not already set in this session)
API_TOKEN = os.environ.get("OPENWEBUI_API_KEY")
API_URL = os.environ["OPENWEBUI_URL"] + "/api/chat/completions"

headers = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}


def classify_emotion_focused(sentence, show_full_response=True, max_retries=3):
    for attempt in range(max_retries):
        data = {
            "model": "llama3.2:3b",
            "messages": [
                {
                    "role": "system",
                    "content": """You are an expert emotion classifier for Bulgarian text.

TASK: Classify into exactly one of these 7 emotions:
happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL - PAY SPECIAL ATTENTION TO:

ANGER indicators:
- Frustration, irritation, annoyance
- Directed at people, situations, or actions
- Words: ядосан, разгневен, раздразнен, бесен, сърдит
- Phrases expressing blame or complaint
- Active, energized negative emotion

DISGUST indicators:
- Physical or moral revulsion
- Rejection, aversion, repulsion
- Words: отвратително, гнусно, противно, мерзко
- Describes something as repellent or nauseating
- Passive withdrawal reaction"""
                },
                {
                    "role": "user",
                    "content": f"""Classify the emotion. Examples:

Ядосан съм на тази ситуация! → anger
Това е отвратително. → disgust
Толкова съм щастлив днес! → happiness
Това ме разстройва много. → sadness
Wow, не очаквах това! → surprise
Страх ме е от това. → fear
Добре. → neutral

Input: {sentence}
Emotion:"""
                }
            ],
            "temperature": 0.05,
            "max_tokens": 20
        }

        try:
            response = requests.post(API_URL, headers=headers, json=data, timeout=60)
            if response.status_code == 200:
                result = response.json()
                full_response = result['choices'][0]['message']['content']

                if show_full_response:
                    print(f"  Response: '{full_response}'")

                extracted_label = extract_emotion_label(full_response)
                return extracted_label, full_response
            else:
                print(f"  Attempt {attempt+1}: Status {response.status_code}")
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
        except Exception as e:
            print(f"  Attempt {attempt+1}: Error - {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
                continue

    return "neutral", "error_fallback"  # Default to neutral instead of error

In [56]:
# Execute with focused prompt
sample_size = 210
sample_df = df.head(sample_size).copy()

predictions = []
full_responses = []

for idx, row in sample_df.iterrows():
    sentence = row['Sentence']
    true_label = row['Label']

    print(f"\n[{idx+1}/{sample_size}]")
    print(f"Input: {sentence[:60]}...")
    print(f"True Label: {true_label}")

    extracted, full_resp = classify_emotion_focused(sentence, show_full_response=False)

    predictions.append(extracted)
    full_responses.append(full_resp)

    print(f"Extracted Label: {extracted}")
    match = "CORRECT" if extracted.lower() == true_label.lower() else "INCORRECT"
    print(f"Result: {match}")

    time.sleep(0.5)

sample_df['predicted_focused'] = predictions
sample_df['full_response_focused'] = full_responses


[1/210]
Input: Ако приемаш моите извинения, просто ме пригърни....
True Label: sadness
Extracted Label: neutral
Result: INCORRECT

[2/210]
Input: Трябвам това, че си ме пригърни....
True Label: sadness
Extracted Label: anger
Result: INCORRECT

[3/210]
Input: С теб всеки миски е прекрасен. Обичам....
True Label: happiness
Extracted Label: happiness
Result: CORRECT

[4/210]
Input: Аз наистина ги гледам по страни и се вълнувам за тези хора....
True Label: happiness
Extracted Label: happiness
Result: CORRECT

[5/210]
Input: Ти избра играта, а аз избрах любовта....
True Label: sadness
Extracted Label: sadness
Result: CORRECT

[6/210]
Input: Не е така. По-скоро бих казал това на теб....
True Label: anger
Extracted Label: anger
Result: CORRECT

[7/210]
Input: Днес ви очаквам, че възможността си възможност,...
True Label: neutral
Extracted Label: neutral
Result: CORRECT

[8/210]
Input: аз избрах това на теб....
True Label: happiness
Extracted Label: neutral
Result: INCORRECT

[9/210]
Input: Д

In [57]:
# Evaluation
y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted_focused'].str.lower()

f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
matches = (y_true == y_pred).sum()
accuracy = matches / len(sample_df)

print("\n" + "=" * 60)
print("RESULTS - FOCUSED PROMPT")
print("=" * 60)
print(f"Sample Size: {sample_size}")
print(f"Correct Predictions: {matches}/{sample_size}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")
print(f"Target F1 Weighted: 0.85")
print(f"Gap to Target: {0.85 - f1_weighted:.4f}")


RESULTS - FOCUSED PROMPT
Sample Size: 210
Correct Predictions: 93/210
Accuracy: 0.4429
F1 Score (Macro): 0.2996
F1 Score (Weighted): 0.4196
Target F1 Weighted: 0.85
Gap to Target: 0.4304


In [58]:
# Compare with previous best
print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"Previous F1 Weighted: 0.5503")
print(f"Current F1 Weighted:  {f1_weighted:.4f}")
print(f"Change: {f1_weighted - 0.5503:+.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, zero_division=0))


COMPARISON
Previous F1 Weighted: 0.5503
Current F1 Weighted:  0.4196
Change: -0.1307

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger       0.11      0.40      0.17        10
     disgust       0.12      0.17      0.14         6
        fear       0.88      0.41      0.56        17
   happiness       0.78      0.18      0.29        39
     neutral       0.50      0.74      0.60        88
     sadness       0.53      0.25      0.34        36
    surprise       0.00      0.00      0.00        14

    accuracy                           0.44       210
   macro avg       0.42      0.31      0.30       210
weighted avg       0.52      0.44      0.42       210



In [59]:
# Check anger and disgust specifically
anger_results = sample_df[sample_df['Label'].str.lower() == 'anger']
disgust_results = sample_df[sample_df['Label'].str.lower() == 'disgust']

print("\n" + "=" * 60)
print("ANGER CLASSIFICATION ANALYSIS")
print("=" * 60)
print(f"Total anger samples: {len(anger_results)}")
if len(anger_results) > 0:
    anger_correct = (anger_results['Label'].str.lower() == anger_results['predicted_focused'].str.lower()).sum()
    print(f"Correctly classified: {anger_correct}/{len(anger_results)}")
    print(f"Anger predictions: {anger_results['predicted_focused'].value_counts().to_dict()}")

print("\n" + "=" * 60)
print("DISGUST CLASSIFICATION ANALYSIS")
print("=" * 60)
print(f"Total disgust samples: {len(disgust_results)}")
if len(disgust_results) > 0:
    disgust_correct = (disgust_results['Label'].str.lower() == disgust_results['predicted_focused'].str.lower()).sum()
    print(f"Correctly classified: {disgust_correct}/{len(disgust_results)}")
    print(f"Disgust predictions: {disgust_results['predicted_focused'].value_counts().to_dict()}")

sample_df.to_csv("focused_results.csv", index=False)
print("\nResults saved to focused_results.csv")


ANGER CLASSIFICATION ANALYSIS
Total anger samples: 10
Correctly classified: 4/10
Anger predictions: {'anger': 4, 'neutral': 3, 'sadness': 2, 'happiness': 1}

DISGUST CLASSIFICATION ANALYSIS
Total disgust samples: 6
Correctly classified: 1/6
Disgust predictions: {'anger': 3, 'neutral': 2, 'disgust': 1}

Results saved to focused_results.csv


In [60]:
# Check for errors in predictions
print(f"Total predictions: {len(predictions)}")
print(f"Error count: {predictions.count('error')}")
print(f"Sample size variable: {sample_size}")
print(f"Actual sample_df size: {len(sample_df)}")

# Show which samples had errors
error_indices = [i for i, pred in enumerate(predictions) if pred == 'error']
if error_indices:
    print(f"\nErrors at indices: {error_indices}")
    for idx in error_indices[:5]:  # Show first 5
        print(f"  Row {idx}: {sample_df.iloc[idx]['Sentence'][:50]}...")

Total predictions: 210
Error count: 0
Sample size variable: 210
Actual sample_df size: 210


## Iteration 6: Few-shot on full dataset

In [61]:
def classify_emotion_fewshot(sentence, show_full_response=False):
    """
    Few-Shot prompt - proven best performer (F1 Weighted: 0.5725 on 42 samples)
    """
    data = {
        "model": "llama3.2:3b",
        "messages": [
            {
                "role": "system",
                "content": """You are an expert emotion classifier for Bulgarian text.
Classify into one of these emotions ONLY: happiness, sadness, anger, surprise, fear, disgust, neutral."""
            },
            {
                "role": "user",
                "content": f"""Classify the emotion in Bulgarian sentences. Examples:

Sentence: Толкова съм щастлив днес!
Emotion: happiness

Sentence: Това ме разстройва много.
Emotion: sadness

Sentence: Ядосан съм на тази ситуация!
Emotion: anger

Sentence: Wow, не очаквах това!
Emotion: surprise

Sentence: Страх ме е от това.
Emotion: fear

Sentence: Това е отвратително.
Emotion: disgust

Sentence: Добре.
Emotion: neutral

Now classify:
Sentence: {sentence}
Emotion: """
            }
        ],
        "temperature": 0.1,
        "max_tokens": 100
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=60)
        if response.status_code == 200:
            result = response.json()
            full_response = result['choices'][0]['message']['content']

            if show_full_response:
                print(f"  Response: '{full_response}'")

            extracted_label = extract_emotion_label(full_response)
            return extracted_label, full_response
        else:
            return "neutral", "error"
    except Exception as e:
        print(f"  Error: {e}")
        return "neutral", "error"

# Run Few-Shot on full dataset
print("=" * 60)
print("FEW-SHOT PROMPT - FULL DATASET (1007 samples)")
print("=" * 60)

sample_size = 1007
sample_df = df.head(sample_size).copy()

print(f"Dataset size: {len(sample_df)}")
print("Starting classification (estimated time: 10-15 minutes)")

predictions = []
full_responses = []

for idx, row in sample_df.iterrows():
    sentence = row['Sentence']

    extracted, full_resp = classify_emotion_fewshot(sentence, show_full_response=False)

    predictions.append(extracted)
    full_responses.append(full_resp)

    if (idx + 1) % 100 == 0:
        print(f"Progress: {idx+1}/{sample_size}")

    time.sleep(0.5)

sample_df['predicted_fewshot'] = predictions
sample_df['full_response_fewshot'] = full_responses

# Evaluation
y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted_fewshot'].str.lower()

f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
matches = (y_true == y_pred).sum()
accuracy = matches / len(sample_df)

print("\n" + "=" * 60)
print("FINAL RESULTS - FEW-SHOT ON FULL DATASET")
print("=" * 60)
print(f"Sample Size: {sample_size}")
print(f"Correct Predictions: {matches}/{sample_size}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")
print(f"Target F1 Weighted: 0.85")
print(f"Gap to Target: {0.85 - f1_weighted:.4f}")

if f1_weighted >= 0.85:
    print("\nTARGET ACHIEVED!")
else:
    print(f"\nCurrent: {f1_weighted:.4f} | Need: {0.85 - f1_weighted:.4f} more")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, zero_division=0))

sample_df.to_csv("fewshot_full_dataset_results.csv", index=False)
print("\nResults saved to fewshot_full_dataset_results.csv")

# Summary comparison
print("\n" + "=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print("Prompt Type    | Samples | F1 Weighted")
print("-" * 60)
print(f"Few-Shot       | 42      | 0.5725")
print(f"Few-Shot       | 1007    | {f1_weighted:.4f}")

FEW-SHOT PROMPT - FULL DATASET (1007 samples)
Dataset size: 1005
Starting classification (estimated time: 10-15 minutes)
Progress: 100/1007
Progress: 200/1007
Progress: 300/1007
Progress: 400/1007
Progress: 500/1007
Progress: 600/1007
Progress: 700/1007
Progress: 800/1007
Progress: 900/1007
Progress: 1000/1007

FINAL RESULTS - FEW-SHOT ON FULL DATASET
Sample Size: 1007
Correct Predictions: 575/1007
Accuracy: 0.5721
F1 Score (Macro): 0.3678
F1 Score (Weighted): 0.5579
Target F1 Weighted: 0.85
Gap to Target: 0.2921

Current: 0.5579 | Need: 0.2921 more

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger       0.17      0.22      0.19        51
     disgust       0.13      0.09      0.11        22
        fear       0.72      0.20      0.31        66
   happiness       0.68      0.52      0.59       187
     neutral       0.63      0.77      0.69       452
     sadness       0.52      0.52      0.52       188
    surprise       0.23      0.13      0.1

# Switch to Llama 3.3 70B model

## Iteration 7: Testing Few-Shot Prompt with 70B Model

In [31]:
def classify_emotion_fewshot_70b(sentence, show_full_response=False):
    """
    Few-Shot prompt with 70B model for improved performance.
    """
    data = {
        "model": "meta-llama/Llama-3.3-70B-Instruct",
        "messages": [
            {
                "role": "system",
                "content": """You are an expert emotion classifier for Bulgarian text.
Classify into one of these emotions ONLY: happiness, sadness, anger, surprise, fear, disgust, neutral."""
            },
            {
                "role": "user",
                "content": f"""Classify the emotion in Bulgarian sentences. Examples:

Sentence: Толкова съм щастлив днес!
Emotion: happiness

Sentence: Това ме разстройва много.
Emotion: sadness

Sentence: Ядосан съм на тази ситуация!
Emotion: anger

Sentence: Wow, не очаквах това!
Emotion: surprise

Sentence: Страх ме е от това.
Emotion: fear

Sentence: Това е отвратително.
Emotion: disgust

Sentence: Добре.
Emotion: neutral

Now classify:
Sentence: {sentence}
Emotion: """
            }
        ],
        "temperature": 0.1,
        "max_tokens": 100
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=90)  # Longer timeout for 70B
        if response.status_code == 200:
            result = response.json()
            full_response = result['choices'][0]['message']['content']

            if show_full_response:
                print(f"  Response: '{full_response}'")

            extracted_label = extract_emotion_label(full_response)
            return extracted_label, full_response
        else:
            return "neutral", "error"
    except Exception as e:
        print(f"  Error: {e}")
        return "neutral", "error"

# Test on small sample first (50 samples)
print("=" * 60)
print("TESTING 70B MODEL ON 50 SAMPLES")
print("=" * 60)

test_size = 50
test_df = df.head(test_size).copy()

print(f"Testing {test_size} samples with 70B model")
print("Note: 70B model is slower but more accurate")

predictions_70b = []
full_responses_70b = []

for idx, row in test_df.iterrows():
    sentence = row['Sentence']

    extracted, full_resp = classify_emotion_fewshot_70b(sentence, show_full_response=False)

    predictions_70b.append(extracted)
    full_responses_70b.append(full_resp)

    if (idx + 1) % 10 == 0:
        print(f"Progress: {idx+1}/{test_size}")

    time.sleep(1)  # Longer delay for 70B

test_df['predicted_70b'] = predictions_70b

# Evaluate
y_true = test_df['Label'].str.lower()
y_pred = test_df['predicted_70b'].str.lower()

f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print("\n" + "=" * 60)
print("70B MODEL - TEST RESULTS (50 samples)")
print("=" * 60)
print(f"F1 Weighted: {f1_weighted:.4f}")
print(f"3B Model (50 samples): ~0.55")
print(f"Improvement: {f1_weighted - 0.55:+.4f}")

if f1_weighted > 0.65:
    print("\n70B shows improvement - proceed with full dataset")
else:
    print("\n70B not showing significant improvement")

print("\n" + classification_report(y_true, y_pred, zero_division=0))

TESTING 70B MODEL ON 50 SAMPLES
Testing 50 samples with 70B model
Note: 70B model is slower but more accurate
emotion: happiness', found 'happiness'.
emotion: sadness', found 'sadness'.
emotion: happiness', found 'happiness'.
emotion: happiness', found 'happiness'.
emotion: sadness 

(this sentence implies a sense of loss or longing, as the speaker chose love, but the other person chose something else, suggesting a potential separation or mismatch.)', found 'sadness'.
emotion: anger', found 'anger'.
emotion: neutral', found 'neutral'.
emotion: neutral', found 'neutral'.

(the sentence appears to be a statement of fact or information, without expressing any strong emotions.)', found 'neutral'.
emotion: neutral', found 'neutral'.
Progress: 10/50
emotion: surprise 

(note: the sentence "някой да ме ощипе" is a common bulgarian idiomatic expression that translates to "pinch me" and is used to express surprise or incredulity, similar to the english phrase "pinch me, i'm dreaming.")', found 

## Iteration 8: Few shot prompt on full dataset with 70B

In [32]:
# Run 70B model on full dataset
print("=" * 60)
print("70B MODEL - FULL DATASET (1005 samples)")
print("=" * 60)

sample_size = 1005
sample_df = df.head(sample_size).copy()

print(f"Dataset size: {len(sample_df)}")
print("Starting classification...")

predictions_70b_full = []
full_responses_70b_full = []

start_time = time.time()

for idx, row in sample_df.iterrows():
    sentence = row['Sentence']

    extracted, full_resp = classify_emotion_fewshot_70b(sentence, show_full_response=False)

    predictions_70b_full.append(extracted)
    full_responses_70b_full.append(full_resp)

    if (idx + 1) % 100 == 0:
        elapsed = (time.time() - start_time) / 60
        avg_time = elapsed / (idx + 1)
        remaining = avg_time * (sample_size - (idx + 1))
        print(f"Progress: {idx+1}/{sample_size} | Elapsed: {elapsed:.1f}min | Est. remaining: {remaining:.1f}min")

    time.sleep(0.5)

sample_df['predicted_70b'] = predictions_70b_full
sample_df['full_response_70b'] = full_responses_70b_full

# Evaluation
y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted_70b'].str.lower()

f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
matches = (y_true == y_pred).sum()
accuracy = matches / len(sample_df)

print("\n" + "=" * 60)
print("FINAL RESULTS - 70B MODEL ON FULL DATASET")
print("=" * 60)
print(f"Sample Size: {sample_size}")
print(f"Correct Predictions: {matches}/{sample_size}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")
print(f"Target F1 Weighted: 0.85")
print(f"Gap to Target: {0.85 - f1_weighted:.4f}")

if f1_weighted >= 0.85:
    print("\n*** TARGET ACHIEVED ***")
else:
    print(f"\nCurrent: {f1_weighted:.4f} | Need: {0.85 - f1_weighted:.4f} more")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, zero_division=0))

print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print("Model | Samples | F1 Weighted | Status")
print("-" * 60)
print(f"3B    | 1005    | 0.5541      | Baseline")
print(f"70B   | 50      | 0.6733      | Test")
print(f"70B   | 1005    | {f1_weighted:.4f}      | Final")

sample_df.to_csv("fewshot_70b_full_dataset_results.csv", index=False)
print("\nResults saved to fewshot_70b_full_dataset_results.csv")

total_time = (time.time() - start_time) / 60
print(f"\nTotal processing time: {total_time:.1f} minutes")

70B MODEL - FULL DATASET (1005 samples)
Estimated time: 20-25 minutes
Dataset size: 1005
Starting classification...
emotion: happiness', found 'happiness'.
emotion: sadness', found 'sadness'.
emotion: happiness', found 'happiness'.
emotion: happiness', found 'happiness'.
emotion: sadness 

(this sentence implies a sense of loss or longing, as the speaker chose love, but the other person chose something else, suggesting a potential separation or mismatch.)', found 'sadness'.
emotion: anger', found 'anger'.
emotion: neutral', found 'neutral'.
emotion: neutral', found 'neutral'.

(the sentence appears to be a statement of fact or information, without expressing any strong emotions.)', found 'neutral'.
emotion: neutral', found 'neutral'.
emotion: surprise 

(note: the sentence "някой да ме ощипе" is a colloquial way of expressing surprise in bulgarian, similar to "pinch me" in english, implying that something is hard to believe or unexpected.)', found 'surprise'.
emotion: surprise', found 

## Iteration 9: Refining Prompt and Temperature for 70B

In [55]:
def classify_emotion_fewshot_70b_refined(sentence, show_full_response=False):
    """
    70B model with lower temperature and refined prompt.
    """
    data = {
        "model": "meta-llama/Llama-3.3-70B-Instruct",
        "messages":
            {
                "role": "system",
                "content": """You are an expert emotion classifier for Bulgarian text.

You MUST classify into EXACTLY ONE of these 7 emotions:
happiness, sadness, anger, surprise, fear, disgust, neutral

Rules:
- Choose the dominant emotion if multiple are present
- Respond with format "Emotion: [label]"
- Use only the 7 emotions listed above"""
            },
            {
                "role": "user",
                "content": f"""Classify the emotion in Bulgarian sentences. Examples:

Sentence: Толкова съм щастлив днес!
Emotion: happiness

Sentence: Това ме разстройва много.
Emotion: sadness

Sentence: Ядосан съм на тази ситуация!
Emotion: anger

Sentence: Wow, не очаквах това!
Emotion: surprise

Sentence: Страх ме е от това.
Emotion: fear

Sentence: Това е отвратително.
Emotion: disgust

Sentence: Добре.
Emotion: neutral

Now classify:
Sentence: {sentence}
Emotion: """
            }
        ],
        "temperature": 0.01,  # Much lower for consistency
        "max_tokens": 20       # Force concise output
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=90)
        if response.status_code == 200:
            result = response.json()
            full_response = result['choices'][0]['message']['content']

            if show_full_response:
                print(f"  Response: '{full_response}'")

            extracted_label = extract_emotion_label(full_response)
            return extracted_label, full_response
        else:
            return "neutral", "error"
    except Exception as e:
        print(f"  Error: {e}")
        return "neutral", "error"

# Test on 100 samples
print("=" * 60)
print("TESTING REFINED 70B (temp=0.01) ON 100 SAMPLES")
print("=" * 60)

test_size = 105
test_df = df.head(test_size).copy()

predictions_refined = []

for idx, row in test_df.iterrows():
    sentence = row['Sentence']
    extracted, _ = classify_emotion_fewshot_70b_refined(sentence, show_full_response=False)
    predictions_refined.append(extracted)

    if (idx + 1) % 25 == 0:
        print(f"Progress: {idx+1}/{test_size}")

    time.sleep(1)

test_df['predicted_refined'] = predictions_refined

y_true = test_df['Label'].str.lower()
y_pred = test_df['predicted_refined'].str.lower()

f1_weighted_refined = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print("\n" + "=" * 60)
print("RESULTS COMPARISON (100 samples)")
print("=" * 60)
print(f"70B temp=0.1:  {0.68:.4f} (estimated)")
print(f"70B temp=0.01: {f1_weighted_refined:.4f}")
print(f"Change: {f1_weighted_refined - 0.68:+.4f}")

if f1_weighted_refined > 0.70:
    print("\nRefined version shows improvement")
else:
    print("\nNo significant improvement from lower temperature")

TESTING REFINED 70B (temp=0.01) ON 100 SAMPLES
Progress: 25/105
Progress: 50/105
Progress: 75/105
Progress: 100/105

RESULTS COMPARISON (100 samples)
70B temp=0.1:  0.6800 (estimated)
70B temp=0.01: 0.6551
Change: -0.0249

No significant improvement from lower temperature


## Iteration 10: Balanced Few-Shot Examples with 70B

In [56]:
def classify_emotion_balanced_fewshot(sentence, show_full_response=False):
    """
    70B model with balanced few-shot examples (multiple per class).
    """
    data = {
        "model": "meta-llama/Llama-3.3-70B-Instruct",
        "messages": [
            {
                "role": "system",
                "content": """You are an expert emotion classifier for Bulgarian text.

Classify into EXACTLY ONE of these 7 emotions:
happiness, sadness, anger, surprise, fear, disgust, neutral"""
            },
            {
                "role": "user",
                "content": f"""Classify Bulgarian text emotions. Multiple examples per emotion:

HAPPINESS examples:
- Толкова съм щастлив днес! → happiness
- Страхотно! → happiness
- Радвам се много! → happiness

SADNESS examples:
- Това ме разстройва много. → sadness
- Тъжен съм. → sadness
- Жал ми е. → sadness

ANGER examples:
- Ядосан съм на тази ситуация! → anger
- Не мога да понасям това! → anger
- Бесен съм! → anger

SURPRISE examples:
- Wow, не очаквах това! → surprise
- Не вярвам! → surprise
- Шокиран съм! → surprise

FEAR examples:
- Страх ме е от това. → fear
- Уплашен съм. → fear
- Тревожа се. → fear

DISGUST examples:
- Това е отвратително. → disgust
- Гадно ми е. → disgust
- Противно! → disgust

NEUTRAL examples:
- Добре. → neutral
- Разбирам. → neutral
- Ясно. → neutral

Now classify:
Input: {sentence}
Emotion:"""
            }
        ],
        "temperature": 0.05,
        "max_tokens": 50
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=90)
        if response.status_code == 200:
            result = response.json()
            full_response = result['choices'][0]['message']['content']

            if show_full_response:
                print(f"  Response: '{full_response}'")

            extracted_label = extract_emotion_label(full_response)
            return extracted_label, full_response
        else:
            return "neutral", "error"
    except Exception as e:
        print(f"  Error: {e}")
        return "neutral", "error"

In [57]:
# Create balanced test set - equal samples per emotion
print("=" * 60)
print("CREATING BALANCED TEST SET")
print("=" * 60)

samples_per_class = 10
balanced_test = []

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    emotion_samples = df[df['Label'].str.lower() == emotion].head(samples_per_class)
    balanced_test.append(emotion_samples)
    print(f"{emotion.capitalize()}: {len(emotion_samples)} samples")

test_df_balanced = pd.concat(balanced_test, ignore_index=True)
print(f"\nTotal balanced test size: {len(test_df_balanced)}")

# Now test with balanced few-shot prompt on balanced data
print("\n" + "=" * 60)
print("TESTING BALANCED FEW-SHOT ON BALANCED DATA")
print("=" * 60)

predictions_balanced = []

for idx, row in test_df_balanced.iterrows():
    sentence = row['Sentence']
    extracted, _ = classify_emotion_balanced_fewshot(sentence, show_full_response=False)
    predictions_balanced.append(extracted)

    if (idx + 1) % 10 == 0:
        print(f"Progress: {idx+1}/{len(test_df_balanced)}")

    time.sleep(1)

test_df_balanced['predicted'] = predictions_balanced

y_true = test_df_balanced['Label'].str.lower()
y_pred = test_df_balanced['predicted'].str.lower()

f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

print("\n" + "=" * 60)
print("BALANCED TEST RESULTS")
print("=" * 60)
print(f"Test set: 10 samples per class (70 total)")
print(f"F1 Weighted: {f1_weighted:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")

print("\n" + classification_report(y_true, y_pred, zero_division=0))

# Per-class accuracy
print("\n" + "=" * 60)
print("PER-CLASS ACCURACY")
print("=" * 60)

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    subset = test_df_balanced[test_df_balanced['Label'].str.lower() == emotion]
    if len(subset) > 0:
        correct = (subset['Label'].str.lower() == subset['predicted'].str.lower()).sum()
        accuracy = correct / len(subset)
        print(f"{emotion.capitalize():12} {correct}/{len(subset)} ({accuracy:.2%})")

CREATING BALANCED TEST SET
Happiness: 10 samples
Sadness: 10 samples
Anger: 10 samples
Surprise: 10 samples
Fear: 10 samples
Disgust: 10 samples
Neutral: 10 samples

Total balanced test size: 70

TESTING BALANCED FEW-SHOT ON BALANCED DATA
Progress: 10/70
Progress: 20/70
Progress: 30/70
Progress: 40/70
Progress: 50/70
Progress: 60/70
Progress: 70/70

BALANCED TEST RESULTS
Test set: 10 samples per class (70 total)
F1 Weighted: 0.6253
F1 Macro: 0.6253

              precision    recall  f1-score   support

       anger       0.50      0.40      0.44        10
     disgust       1.00      0.50      0.67        10
        fear       1.00      0.70      0.82        10
   happiness       0.78      0.70      0.74        10
     neutral       0.37      1.00      0.54        10
     sadness       0.67      0.60      0.63        10
    surprise       0.80      0.40      0.53        10

    accuracy                           0.61        70
   macro avg       0.73      0.61      0.63        70
weig

## Iteration 11: Reduced Samples per Class for Balanced Few-Shot

In [61]:
# Create balanced test set - equal samples per emotion
print("=" * 60)
print("CREATING BALANCED TEST SET")
print("=" * 60)

samples_per_class = 7
balanced_test = []

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    emotion_samples = df[df['Label'].str.lower() == emotion].head(samples_per_class)
    balanced_test.append(emotion_samples)
    print(f"{emotion.capitalize()}: {len(emotion_samples)} samples")

test_df_balanced = pd.concat(balanced_test, ignore_index=True)
print(f"\nTotal balanced test size: {len(test_df_balanced)}")

# Now test with balanced few-shot prompt on balanced data
print("\n" + "=" * 60)
print("TESTING BALANCED FEW-SHOT ON BALANCED DATA")
print("=" * 60)

predictions_balanced = []

for idx, row in test_df_balanced.iterrows():
    sentence = row['Sentence']
    extracted, _ = classify_emotion_balanced_fewshot(sentence, show_full_response=False)
    predictions_balanced.append(extracted)

    if (idx + 1) % 10 == 0:
        print(f"Progress: {idx+1}/{len(test_df_balanced)}")

    time.sleep(1)

test_df_balanced['predicted'] = predictions_balanced

y_true = test_df_balanced['Label'].str.lower()
y_pred = test_df_balanced['predicted'].str.lower()

f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

print("\n" + "=" * 60)
print("BALANCED TEST RESULTS")
print("=" * 60)
print(f"Test set: 10 samples per class (70 total)")
print(f"F1 Weighted: {f1_weighted:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")

print("\n" + classification_report(y_true, y_pred, zero_division=0))

# Per-class accuracy
print("\n" + "=" * 60)
print("PER-CLASS ACCURACY")
print("=" * 60)

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    subset = test_df_balanced[test_df_balanced['Label'].str.lower() == emotion]
    if len(subset) > 0:
        correct = (subset['Label'].str.lower() == subset['predicted'].str.lower()).sum()
        accuracy = correct / len(subset)
        print(f"{emotion.capitalize():12} {correct}/{len(subset)} ({accuracy:.2%})")

CREATING BALANCED TEST SET
Happiness: 7 samples
Sadness: 7 samples
Anger: 7 samples
Surprise: 7 samples
Fear: 7 samples
Disgust: 7 samples
Neutral: 7 samples

Total balanced test size: 49

TESTING BALANCED FEW-SHOT ON BALANCED DATA
Progress: 10/49
Progress: 20/49
Progress: 30/49
Progress: 40/49

BALANCED TEST RESULTS
Test set: 10 samples per class (70 total)
F1 Weighted: 0.6196
F1 Macro: 0.6196

              precision    recall  f1-score   support

       anger       0.50      0.43      0.46         7
     disgust       1.00      0.43      0.60         7
        fear       1.00      0.71      0.83         7
   happiness       0.67      0.57      0.62         7
     neutral       0.39      1.00      0.56         7
     sadness       0.62      0.71      0.67         7
    surprise       1.00      0.43      0.60         7

    accuracy                           0.61        49
   macro avg       0.74      0.61      0.62        49
weighted avg       0.74      0.61      0.62        49


PER

## Iteration 12: Anti-Neutral Bias Prompt with 70B

In [62]:
import requests
import json
import os
import time
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import re

# API Configuration
API_TOKEN = os.environ.get("OPENWEBUI_API_KEY")
API_URL = os.environ["OPENWEBUI_URL"] + "/api/chat/completions"

headers = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}

def extract_emotion_label(response_text):
    """
    Extract emotion label from LLM response.
    """
    if not response_text:
        return "neutral"

    response_text = response_text.lower().strip()
    valid_emotions = ["happiness", "sadness", "anger", "surprise", "fear", "disgust", "neutral"]

    emotion_pattern = r'emotion:\s*(\w+)'
    match = re.search(emotion_pattern, response_text, re.IGNORECASE)
    if match:
        extracted = match.group(1).lower()
        if extracted in valid_emotions:
            return extracted

    first_line = response_text.split('\n')[0].strip()
    for emotion in valid_emotions:
        if emotion in first_line:
            return emotion

    for emotion in valid_emotions:
        if emotion in response_text:
            return emotion

    return "neutral"

def classify_emotion_anti_neutral_bias(sentence, show_full_response=False):
    """
    70B model with explicit instruction to avoid neutral over-prediction.
    """
    data = {
        "model": "meta-llama/Llama-3.3-70B-Instruct",
        "messages": [
            {
                "role": "system",
                "content": """You are an expert emotion classifier for Bulgarian text.

Classify into EXACTLY ONE emotion:
happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL INSTRUCTIONS:
- Only use "neutral" when there is truly NO emotional content
- If ANY emotion is present, classify it (not neutral)
- Prefer specific emotions over neutral when in doubt
- Neutral is for factual statements only, not mild emotions"""
            },
            {
                "role": "user",
                "content": f"""Examples:

HAPPINESS: Толкова съм щастлив днес! | Страхотно! | Радвам се много!
SADNESS: Това ме разстройва много. | Тъжен съм. | Жал ми е.
ANGER: Ядосан съм! | Не мога да понасям това! | Бесен съм!
SURPRISE: Wow, не очаквах това! | Не вярвам! | Шокиран съм!
FEAR: Страх ме е. | Уплашен съм. | Тревожа се.
DISGUST: Това е отвратително. | Гадно ми е. | Противно!
NEUTRAL: Добре. | Разбирам. | Ясно.

Input: {sentence}
Emotion:"""
            }
        ],
        "temperature": 0.01,
        "max_tokens": 10
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=90)
        if response.status_code == 200:
            result = response.json()
            full_response = result['choices'][0]['message']['content']

            if show_full_response:
                print(f"  Response: '{full_response}'")

            extracted_label = extract_emotion_label(full_response)
            return extracted_label, full_response
        else:
            return "neutral", "error"
    except Exception as e:
        print(f"  Error: {e}")
        return "neutral", "error"

# Load data
df = pd.read_csv("Test_set.csv")
df = df.rename(columns={"Sentence": "Sentence", "Emotion_core": "Label"})

# Create balanced test set
print("=" * 60)
print("CREATING BALANCED TEST SET")
print("=" * 60)

samples_per_class = 10
balanced_test = []

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    emotion_samples = df[df['Label'].str.lower() == emotion].head(samples_per_class)
    balanced_test.append(emotion_samples)
    print(f"{emotion.capitalize()}: {len(emotion_samples)} samples")

test_df_balanced = pd.concat(balanced_test, ignore_index=True)
print(f"\nTotal balanced test size: {len(test_df_balanced)}")

# Test anti-neutral bias prompt
print("\n" + "=" * 60)
print("TESTING ANTI-NEUTRAL BIAS PROMPT")
print("=" * 60)

predictions_anti_bias = []

for idx, row in test_df_balanced.iterrows():
    sentence = row['Sentence']
    extracted, _ = classify_emotion_anti_neutral_bias(sentence, show_full_response=False)
    predictions_anti_bias.append(extracted)

    if (idx + 1) % 10 == 0:
        print(f"Progress: {idx+1}/70")

    time.sleep(1)

test_df_balanced['predicted_anti_bias'] = predictions_anti_bias

# Evaluation
y_true = test_df_balanced['Label'].str.lower()
y_pred = test_df_balanced['predicted_anti_bias'].str.lower()

f1_weighted_new = f1_score(y_true, y_pred, average='weighted', zero_division=0)
f1_macro_new = f1_score(y_true, y_pred, average='macro', zero_division=0)

print("\n" + "=" * 60)
print("RESULTS COMPARISON")
print("=" * 60)
print(f"Balanced few-shot:     F1 Weighted 0.6253")
print(f"Anti-neutral bias:     F1 Weighted {f1_weighted_new:.4f}")
print(f"Change: {f1_weighted_new - 0.6253:+.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, zero_division=0))

# Per-class accuracy
print("\n" + "=" * 60)
print("PER-CLASS ACCURACY")
print("=" * 60)

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    subset = test_df_balanced[test_df_balanced['Label'].str.lower() == emotion]
    if len(subset) > 0:
        correct = (subset['Label'].str.lower() == subset['predicted_anti_bias'].str.lower()).sum()
        accuracy = correct / len(subset)
        print(f"{emotion.capitalize():12} {correct}/{len(subset)} ({accuracy:.2%})")

# Check neutral predictions
neutral_pred_count = (test_df_balanced['predicted_anti_bias'].str.lower() == 'neutral').sum()
neutral_true_count = (test_df_balanced['Label'].str.lower() == 'neutral').sum()

print("\n" + "=" * 60)
print("NEUTRAL PREDICTION ANALYSIS")
print("=" * 60)
print(f"True neutral samples: {neutral_true_count}")
print(f"Predicted as neutral: {neutral_pred_count}")
print(f"Reduction from previous: {neutral_pred_count} (was 10+)")

# Save results
test_df_balanced.to_csv("anti_neutral_bias_results.csv", index=False)
print("\nResults saved to anti_neutral_bias_results.csv")

CREATING BALANCED TEST SET
Happiness: 10 samples
Sadness: 10 samples
Anger: 10 samples
Surprise: 10 samples
Fear: 10 samples
Disgust: 10 samples
Neutral: 10 samples

Total balanced test size: 70

TESTING ANTI-NEUTRAL BIAS PROMPT
Progress: 10/70
Progress: 20/70
Progress: 30/70
Progress: 40/70
Progress: 50/70
Progress: 60/70
Progress: 70/70

RESULTS COMPARISON
Balanced few-shot:     F1 Weighted 0.6253
Anti-neutral bias:     F1 Weighted 0.6310
Change: +0.0057

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger       0.38      0.30      0.33        10
     disgust       1.00      0.50      0.67        10
        fear       1.00      0.70      0.82        10
   happiness       0.80      0.80      0.80        10
     neutral       0.44      0.80      0.57        10
     sadness       0.62      0.80      0.70        10
    surprise       0.56      0.50      0.53        10

    accuracy                           0.63        70
   macro avg       0.68     

## Iteration 13: Anti-Neutral Bias Prompt with 70B on Full Balanced Set

In [65]:
import requests
import json
import os
import time
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import re

# API Configuration
API_TOKEN = os.environ.get("OPENWEBUI_API_KEY")
API_URL = os.environ["OPENWEBUI_URL"] + "/api/chat/completions"

headers = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}

def extract_emotion_label(response_text):
    """
    Extract emotion label from LLM response.
    """
    if not response_text:
        return "neutral"

    response_text = response_text.lower().strip()
    valid_emotions = ["happiness", "sadness", "anger", "surprise", "fear", "disgust", "neutral"]

    emotion_pattern = r'emotion:\s*(\w+)'
    match = re.search(emotion_pattern, response_text, re.IGNORECASE)
    if match:
        extracted = match.group(1).lower()
        if extracted in valid_emotions:
            return extracted

    first_line = response_text.split('\n')[0].strip()
    for emotion in valid_emotions:
        if emotion in first_line:
            return emotion

    for emotion in valid_emotions:
        if emotion in response_text:
            return emotion

    return "neutral"

def classify_emotion_anti_neutral_bias(sentence, show_full_response=False):
    """
    70B model with explicit instruction to avoid neutral over-prediction.
    """
    data = {
        "model": "meta-llama/Llama-3.3-70B-Instruct",
        "messages": [
            {
                "role": "system",
                "content": """You are an expert emotion classifier for Bulgarian text.

Classify into EXACTLY ONE emotion:
happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL INSTRUCTIONS:
- Only use "neutral" when there is truly NO emotional content
- If ANY emotion is present, classify it (not neutral)
- Prefer specific emotions over neutral when in doubt
- Neutral is for factual statements only, not mild emotions"""
            },
            {
                "role": "user",
                "content": f"""Examples:

HAPPINESS: Толкова съм щастлив днес! | Страхотно! | Радвам се много!
SADNESS: Това ме разстройва много. | Тъжен съм. | Жал ми е.
ANGER: Ядосан съм! | Не мога да понасям това! | Бесен съм!
SURPRISE: Wow, не очаквах това! | Не вярвам! | Шокиран съм!
FEAR: Страх ме е. | Уплашен съм. | Тревожа се.
DISGUST: Това е отвратително. | Гадно ми е. | Противно!
NEUTRAL: Добре. | Разбирам. | Ясно.

Input: {sentence}
Emotion:"""
            }
        ],
        "temperature": 0.01,
        "max_tokens": 10
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=90)
        if response.status_code == 200:
            result = response.json()
            full_response = result['choices'][0]['message']['content']

            if show_full_response:
                print(f"  Response: '{full_response}'")

            extracted_label = extract_emotion_label(full_response)
            return extracted_label, full_response
        else:
            return "neutral", "error"
    except Exception as e:
        print(f"  Error: {e}")
        return "neutral", "error"

# Load data
df = pd.read_csv("Test_set.csv")
df = df.rename(columns={"Sentence": "Sentence", "Emotion_core": "Label"})

# Create balanced test set with RANDOM sampling
print("=" * 60)
print("CREATING BALANCED TEST SET (RANDOM SAMPLING)")
print("=" * 60)

samples_per_class = 10
balanced_test = []
random_seed = 12

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    emotion_df = df[df['Label'].str.lower() == emotion]

    if len(emotion_df) >= samples_per_class:
        emotion_samples = emotion_df.sample(n=samples_per_class, random_state=random_seed)
    else:
        emotion_samples = emotion_df  # Use all if less than 10

    balanced_test.append(emotion_samples)
    print(f"{emotion.capitalize()}: {len(emotion_samples)} samples (total available: {len(emotion_df)})")

test_df_balanced = pd.concat(balanced_test, ignore_index=True)
print(f"\nTotal balanced test size: {len(test_df_balanced)}")

# Test anti-neutral bias prompt on random balanced sample
print("\n" + "=" * 60)
print("TESTING ANTI-NEUTRAL BIAS PROMPT (RANDOM SAMPLE)")
print("=" * 60)

predictions_anti_bias = []

for idx, row in test_df_balanced.iterrows():
    sentence = row['Sentence']
    extracted, _ = classify_emotion_anti_neutral_bias(sentence, show_full_response=False)
    predictions_anti_bias.append(extracted)

    if (idx + 1) % 10 == 0:
        print(f"Progress: {idx+1}/70")

    time.sleep(1)

test_df_balanced['predicted_anti_bias'] = predictions_anti_bias

# Evaluation
y_true = test_df_balanced['Label'].str.lower()
y_pred = test_df_balanced['predicted_anti_bias'].str.lower()

f1_weighted_random = f1_score(y_true, y_pred, average='weighted', zero_division=0)
f1_macro_random = f1_score(y_true, y_pred, average='macro', zero_division=0)

print("\n" + "=" * 60)
print("RESULTS (RANDOM BALANCED SAMPLE)")
print("=" * 60)
print(f"F1 Weighted: {f1_weighted_random:.4f}")
print(f"F1 Macro: {f1_macro_random:.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, zero_division=0))

# Per-class accuracy
print("\n" + "=" * 60)
print("PER-CLASS ACCURACY")
print("=" * 60)

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    subset = test_df_balanced[test_df_balanced['Label'].str.lower() == emotion]
    if len(subset) > 0:
        correct = (subset['Label'].str.lower() == subset['predicted_anti_bias'].str.lower()).sum()
        accuracy = correct / len(subset)
        print(f"{emotion.capitalize():12} {correct}/{len(subset)} ({accuracy:.2%})")

# Check neutral predictions
neutral_pred_count = (test_df_balanced['predicted_anti_bias'].str.lower() == 'neutral').sum()
neutral_true_count = (test_df_balanced['Label'].str.lower() == 'neutral').sum()

print("\n" + "=" * 60)
print("NEUTRAL PREDICTION ANALYSIS")
print("=" * 60)
print(f"True neutral samples: {neutral_true_count}")
print(f"Predicted as neutral: {neutral_pred_count}")

# Save results
test_df_balanced.to_csv("random_balanced_test_results.csv", index=False)
print("\nResults saved to random_balanced_test_results.csv")

CREATING BALANCED TEST SET (RANDOM SAMPLING)
Happiness: 10 samples (total available: 187)
Sadness: 10 samples (total available: 188)
Anger: 10 samples (total available: 51)
Surprise: 10 samples (total available: 39)
Fear: 10 samples (total available: 66)
Disgust: 10 samples (total available: 22)
Neutral: 10 samples (total available: 452)

Total balanced test size: 70

TESTING ANTI-NEUTRAL BIAS PROMPT (RANDOM SAMPLE)
Progress: 10/70
Progress: 20/70
Progress: 30/70
Progress: 40/70
Progress: 50/70
Progress: 60/70
Progress: 70/70

RESULTS (RANDOM BALANCED SAMPLE)
F1 Weighted: 0.5419
F1 Macro: 0.5419

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger       0.33      0.40      0.36        10
     disgust       1.00      0.40      0.57        10
        fear       1.00      0.30      0.46        10
   happiness       0.89      0.80      0.84        10
     neutral       0.36      1.00      0.53        10
     sadness       0.60      0.60      0.60      

In [70]:
# Analyze correct and incorrect predictions
print("\n" + "=" * 60)
print("DETAILED PREDICTION ANALYSIS")
print("=" * 60)

# Separate correct and incorrect predictions
correct_predictions = test_df_balanced[
    test_df_balanced['Label'].str.lower() == test_df_balanced['predicted_anti_bias'].str.lower()
]

incorrect_predictions = test_df_balanced[
    test_df_balanced['Label'].str.lower() != test_df_balanced['predicted_anti_bias'].str.lower()
]

print(f"\nTotal: {len(test_df_balanced)} samples")
print(f"Correct: {len(correct_predictions)}")
print(f"Incorrect: {len(incorrect_predictions)}")

# Show all correct predictions
print("\n" + "=" * 60)
print("CORRECT PREDICTIONS")
print("=" * 60)

for idx, row in correct_predictions.iterrows():
    print(f"\n[CORRECT] True: {row['Label']} | Predicted: {row['predicted_anti_bias']}")
    print(f"Sentence: {row['Sentence']}")

# Show all incorrect predictions
print("\n" + "=" * 60)
print("INCORRECT PREDICTIONS")
print("=" * 60)

for idx, row in incorrect_predictions.iterrows():
    print(f"\n[WRONG] True: {row['Label']} | Predicted: {row['predicted_anti_bias']}")
    print(f"Sentence: {row['Sentence']}")

# Group incorrect predictions by true emotion
print("\n" + "=" * 60)
print("INCORRECT PREDICTIONS BY EMOTION")
print("=" * 60)

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    emotion_errors = incorrect_predictions[incorrect_predictions['Label'].str.lower() == emotion]

    if len(emotion_errors) > 0:
        print(f"\n{emotion.upper()} - {len(emotion_errors)} errors:")
        print("-" * 60)

        for idx, row in emotion_errors.iterrows():
            print(f"  True: {row['Label']} → Predicted: {row['predicted_anti_bias']}")
            print(f"  Sentence: {row['Sentence']}")
            print()

# Confusion patterns
print("\n" + "=" * 60)
print("MOST COMMON CONFUSION PATTERNS")
print("=" * 60)

confusion_patterns = {}

for idx, row in incorrect_predictions.iterrows():
    true_label = row['Label'].lower()
    pred_label = row['predicted_anti_bias'].lower()
    pattern = f"{true_label} → {pred_label}"

    if pattern not in confusion_patterns:
        confusion_patterns[pattern] = []

    confusion_patterns[pattern].append(row['Sentence'])

# Sort by frequency
sorted_patterns = sorted(confusion_patterns.items(), key=lambda x: len(x[1]), reverse=True)

for pattern, sentences in sorted_patterns:
    print(f"\n{pattern}: {len(sentences)} occurrences")
    for i, sentence in enumerate(sentences, 1):
        print(f"  {i}. {sentence}")

# Summary statistics per emotion
print("\n" + "=" * 60)
print("ERROR RATE BY EMOTION")
print("=" * 60)

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    emotion_total = len(test_df_balanced[test_df_balanced['Label'].str.lower() == emotion])
    emotion_correct = len(correct_predictions[correct_predictions['Label'].str.lower() == emotion])
    emotion_incorrect = len(incorrect_predictions[incorrect_predictions['Label'].str.lower() == emotion])

    if emotion_total > 0:
        error_rate = (emotion_incorrect / emotion_total) * 100
        print(f"{emotion.capitalize():12} Correct: {emotion_correct}/{emotion_total} | Incorrect: {emotion_incorrect}/{emotion_total} | Error Rate: {error_rate:.1f}%")

# Save detailed analysis
correct_predictions.to_csv("correct_predictions_detailed.csv", index=False)
incorrect_predictions.to_csv("incorrect_predictions_detailed.csv", index=False)

print("\n" + "=" * 60)
print("FILES SAVED")
print("=" * 60)
print("correct_predictions_detailed.csv - All correct predictions")
print("incorrect_predictions_detailed.csv - All incorrect predictions")


DETAILED PREDICTION ANALYSIS

Total: 70 samples
Correct: 38
Incorrect: 32

CORRECT PREDICTIONS

[CORRECT] True: happiness | Predicted: happiness
Sentence: Добър апетит!

[CORRECT] True: happiness | Predicted: happiness
Sentence: и Ники си спечелиха

[CORRECT] True: happiness | Predicted: happiness
Sentence: Супер! Много благодарим!

[CORRECT] True: happiness | Predicted: happiness
Sentence: да се чувствам добре,

[CORRECT] True: happiness | Predicted: happiness
Sentence: Много по-добро.

[CORRECT] True: happiness | Predicted: happiness
Sentence: но не спят до човека, когато наистина харесват.

[CORRECT] True: happiness | Predicted: happiness
Sentence: край басейна и да се насладим на времето си

[CORRECT] True: happiness | Predicted: happiness
Sentence: Много по-добро.

[CORRECT] True: sadness | Predicted: sadness
Sentence: защото не ми се тръгва на третия ден,

[CORRECT] True: sadness | Predicted: sadness
Sentence: решението. Тогава си не ме обичаш.

[CORRECT] True: sadness | Predict

## Iteration 14: Anti-Neutral Bias Prompt with 70B on Random Balanced Set

In [37]:
import requests
import json
import os
import time
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import re

# API Configuration
API_TOKEN = os.environ.get("OPENWEBUI_API_KEY")
API_URL = os.environ["OPENWEBUI_URL"] + "/api/chat/completions"

headers = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}

def extract_emotion_label(response_text):
    """
    Extract emotion label from LLM response.
    """
    if not response_text:
        return "neutral"

    response_text = response_text.lower().strip()
    valid_emotions = ["happiness", "sadness", "anger", "surprise", "fear", "disgust", "neutral"]

    emotion_pattern = r'emotion:\s*(\w+)'
    match = re.search(emotion_pattern, response_text, re.IGNORECASE)
    if match:
        extracted = match.group(1).lower()
        if extracted in valid_emotions:
            return extracted

    first_line = response_text.split('\n')[0].strip()
    for emotion in valid_emotions:
        if emotion in first_line:
            return emotion

    for emotion in valid_emotions:
        if emotion in response_text:
            return emotion

    return "neutral"

def classify_emotion_anti_neutral_bias(sentence, show_full_response=False):
    """
    70B model with explicit instruction to avoid neutral over-prediction.
    """
    data = {
        "model": "meta-llama/Llama-3.3-70B-Instruct",
        "messages": [
            {
                "role": "system",
                "content": """You are an expert emotion classifier for Bulgarian text.

Classify into EXACTLY ONE emotion:
happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL INSTRUCTIONS:
- Only use "neutral" when there is truly NO emotional content
- If ANY emotion is present, classify it (not neutral)
- Prefer specific emotions over neutral when in doubt
- Neutral is for factual statements only, not mild emotions"""
            },
            {
                "role": "user",
                "content": f"""Examples:

HAPPINESS: Толкова съм щастлив днес! | Страхотно! | Радвам се много!
SADNESS: Това ме разстройва много. | Тъжен съм. | Жал ми е.
ANGER: Ядосан съм! | Не мога да понасям това! | Бесен съм!
SURPRISE: Wow, не очаквах това! | Не вярвам! | Шокиран съм!
FEAR: Страх ме е. | Уплашен съм. | Тревожа се.
DISGUST: Това е отвратително. | Гадно ми е. | Противно!
NEUTRAL: Добре. | Разбирам. | Ясно.

Input: {sentence}
Emotion:"""
            }
        ],
        "temperature": 0.01,
        "max_tokens": 100
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=90)
        if response.status_code == 200:
            result = response.json()
            full_response = result['choices'][0]['message']['content']

            if show_full_response:
                print(f"  Response: '{full_response}'")

            extracted_label = extract_emotion_label(full_response)
            return extracted_label, full_response
        else:
            return "neutral", "error"
    except Exception as e:
        print(f"  Error: {e}")
        return "neutral", "error"

# Load full dataset
df = pd.read_csv("Test_set.csv")
df = df.rename(columns={"Sentence": "Sentence", "Emotion_core": "Label"})

print("=" * 60)
print("ANTI-NEUTRAL BIAS - FULL DATASET (UNBALANCED)")
print("=" * 60)
print(f"Total dataset size: {len(df)}")
print("\nClass distribution:")
print(df['Label'].value_counts())

# Run on full dataset
sample_size = len(df)
sample_df = df.copy()

print("\n" + "=" * 60)
print("STARTING CLASSIFICATION")
print("=" * 60)
print(f"Classifying {sample_size} samples")
print("Estimated time: 25-30 minutes")

predictions_full = []
full_responses_full = []

start_time = time.time()

for idx, row in sample_df.iterrows():
    sentence = row['Sentence']

    extracted, full_resp = classify_emotion_anti_neutral_bias(sentence, show_full_response=False)

    predictions_full.append(extracted)
    full_responses_full.append(full_resp)

    if (idx + 1) % 100 == 0:
        elapsed = (time.time() - start_time) / 60
        avg_time = elapsed / (idx + 1)
        remaining = avg_time * (sample_size - (idx + 1))
        print(f"Progress: {idx+1}/{sample_size} | Elapsed: {elapsed:.1f}min | Est. remaining: {remaining:.1f}min")

    time.sleep(1)

sample_df['predicted'] = predictions_full
sample_df['full_response'] = full_responses_full

# Evaluation
y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted'].str.lower()

f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
matches = (y_true == y_pred).sum()
accuracy = matches / len(sample_df)

print("\n" + "=" * 60)
print("FINAL RESULTS - FULL DATASET (UNBALANCED)")
print("=" * 60)
print(f"Sample Size: {sample_size}")
print(f"Correct Predictions: {matches}/{sample_size}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")
print(f"Target F1 Weighted: 0.85")
print(f"Gap to Target: {0.85 - f1_weighted:.4f}")

if f1_weighted >= 0.85:
    print("\n*** TARGET ACHIEVED ***")
else:
    print(f"\nCurrent: {f1_weighted:.4f} | Need: {0.85 - f1_weighted:.4f} more")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, zero_division=0))

# Neutral prediction analysis
neutral_pred = (sample_df['predicted'].str.lower() == 'neutral').sum()
neutral_true = (sample_df['Label'].str.lower() == 'neutral').sum()

print("\n" + "=" * 60)
print("NEUTRAL PREDICTION ANALYSIS")
print("=" * 60)
print(f"True neutral samples: {neutral_true}")
print(f"Predicted as neutral: {neutral_pred}")
print(f"Difference: {neutral_pred - neutral_true}")

# Per-class results
print("\n" + "=" * 60)
print("PER-CLASS RESULTS")
print("=" * 60)

for emotion in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']:
    emotion_total = (sample_df['Label'].str.lower() == emotion).sum()
    emotion_correct = ((sample_df['Label'].str.lower() == emotion) &
                       (sample_df['predicted'].str.lower() == emotion)).sum()

    if emotion_total > 0:
        emotion_accuracy = (emotion_correct / emotion_total) * 100
        print(f"{emotion.capitalize():12} Total: {emotion_total:4} | Correct: {emotion_correct:4} | Accuracy: {emotion_accuracy:5.1f}%")

# Save results
sample_df.to_csv("anti_neutral_bias_full_unbalanced.csv", index=False)
print("RESULTS SAVED")
print("File: anti_neutral_bias_full_unbalanced.csv")

total_time = (time.time() - start_time) / 60
print(f"\nTotal processing time: {total_time:.1f} minutes")

# Comparison with previous results
print("ALL EXPERIMENTS SUMMARY")
print("Experiment                    | Samples | F1 Weighted | F1 Macro")
print(f"Few-Shot (3B)                 | 1005    | 0.5541      | 0.3655")
print(f"Few-Shot (70B)                | 1005    | 0.6791      | 0.5637")
print(f"Anti-Neutral (70B) Unbalanced | {sample_size:<7} | {f1_weighted:.4f}      | {f1_macro:.4f}")

print("\n" + "=" * 60)
print("CONCLUSION")
print("=" * 60)

if f1_weighted >= 0.85:
    print("Target F1 Weighted >= 0.85 achieved!")
    print("Task complete.")
elif f1_weighted >= 0.70:
    print(f"Achieved F1 Weighted: {f1_weighted:.4f}")
    print(f"Gap to target: {0.85 - f1_weighted:.4f}")
    print("\nFurther improvements may require:")
    print("- Fine-tuning the model on Bulgarian emotion data")
    print("- Using more sophisticated prompt engineering techniques")
    print("- Ensemble methods combining multiple prompts")
else:
    print(f"Current F1 Weighted: {f1_weighted:.4f}")
    print("Prompt engineering limits reached for this model")

ANTI-NEUTRAL BIAS - FULL DATASET (UNBALANCED)
Total dataset size: 1005

Class distribution:
Label
neutral      452
sadness      188
happiness    187
fear          66
anger         51
surprise      39
disgust       22
Name: count, dtype: int64

STARTING CLASSIFICATION
Classifying 1005 samples
Estimated time: 25-30 minutes
Progress: 100/1005 | Elapsed: 2.8min | Est. remaining: 25.2min
Progress: 200/1005 | Elapsed: 5.9min | Est. remaining: 23.7min
Progress: 300/1005 | Elapsed: 8.8min | Est. remaining: 20.7min
Progress: 400/1005 | Elapsed: 11.4min | Est. remaining: 17.2min
Progress: 500/1005 | Elapsed: 13.9min | Est. remaining: 14.0min
Progress: 600/1005 | Elapsed: 16.3min | Est. remaining: 11.0min
Progress: 700/1005 | Elapsed: 18.9min | Est. remaining: 8.2min
Progress: 800/1005 | Elapsed: 21.7min | Est. remaining: 5.6min
Progress: 900/1005 | Elapsed: 24.3min | Est. remaining: 2.8min
Progress: 1000/1005 | Elapsed: 26.8min | Est. remaining: 0.1min

FINAL RESULTS - FULL DATASET (UNBALANCED)


## Iteration 15: Majority vote with Anti-Neutral Bias Prompt on 70B

In [33]:
def classify_emotion_self_consistency(sentence, num_samples=3, show_full_response=False):
    """
    Self-consistency: Run multiple inferences and take majority vote.
    This is a legitimate technique from the prompting guide.
    """
    predictions = []

    for i in range(num_samples):
        data = {
            "model": "meta-llama/Llama-3.3-70B-Instruct",
            "messages": [
                {
                    "role": "system",
                    "content": """You are an expert emotion classifier for Bulgarian text.

Classify into EXACTLY ONE emotion:
happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL INSTRUCTIONS:
- Only use "neutral" when there is truly NO emotional content
- If ANY emotion is present, classify it (not neutral)
- Prefer specific emotions over neutral when in doubt"""
                },
                {
                    "role": "user",
                    "content": f"""Examples:

HAPPINESS: Толкова съм щастлив днес!
SADNESS: Това ме разстройва много.
ANGER: Ядосан съм!
SURPRISE: Wow, не очаквах това!
FEAR: Страх ме е.
DISGUST: Това е отвратително.
NEUTRAL: Добре.

Input: {sentence}
Emotion:"""
                }
            ],
            "temperature": 0.3,
            "max_tokens": 100
        }

        try:
            response = requests.post(API_URL, headers=headers, json=data, timeout=90)
            if response.status_code == 200:
                result = response.json()
                full_response = result['choices'][0]['message']['content']
                extracted_label = extract_emotion_label(full_response)
                predictions.append(extracted_label)
            else:
                predictions.append("neutral")
        except Exception as e:
            predictions.append("neutral")

    # Majority vote
    from collections import Counter
    vote_counts = Counter(predictions)
    majority_emotion = vote_counts.most_common(1)[0][0]

    if show_full_response:
        print(f"  Votes: {predictions} → {majority_emotion}")

    return majority_emotion, str(predictions)

# Test on 50 samples
print("SELF-CONSISTENCY TEST (50 samples)")
print("Running 3 inferences per sample and taking majority vote")
print("This is a legitimate technique from the prompting guide")

test_size = 210
random_seed = 12
test_df = df.head(test_size).copy()

predictions_sc = []

start_time = time.time()

for idx, row in test_df.iterrows():
    sentence = row['Sentence']
    extracted, votes = classify_emotion_self_consistency(sentence, num_samples=3, show_full_response=False)
    predictions_sc.append(extracted)

    if (idx + 1) % 10 == 0:
        print(f"Progress: {idx+1}/{test_size}")

    time.sleep(1)

test_df['predicted_sc'] = predictions_sc

y_true = test_df['Label'].str.lower()
y_pred = test_df['predicted_sc'].str.lower()

f1_weighted_sc = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print("SELF-CONSISTENCY RESULTS")
print(f"Previous (single inference): F1 Weighted ~0.68")
print(f"Self-consistency (3 votes):  F1 Weighted {f1_weighted_sc:.4f}")
print(f"Change: {f1_weighted_sc - 0.68:+.4f}")

print("\n" + classification_report(y_true, y_pred, zero_division=0))

SELF-CONSISTENCY TEST (50 samples)
Running 3 inferences per sample and taking majority vote
This is a legitimate technique from the prompting guide

(текстът изразява съжаление и тъга, което е характерно за емоцията на тъга.)', found 'sadness'.

(текстът изразява съжаление и тъга, което е характерно за емоцията на тъга.)', found 'sadness'.

(текстът изразява съжаление и тъга, което е характерно за емоцията на тъга.)', found 'sadness'.

(reasoning: the sentence implies a need for comfort, which is often associated with feelings of sadness or emotional distress. the request to be hugged suggests a desire for emotional support, which is typically sought in times of sadness or vulnerability.)', found 'sadness'.

(reasoning: the sentence implies a contrast between two choices, where the speaker chose love, suggesting a potential longing or melancholy tone, which aligns more closely with sadness.)', found 'sadness'.

(reasoning: the sentence implies a contrast between two choices, where the 

## Iteration 16: Majority vote with Anti-Neutral Bias Prompt on 70B - Full Dataset

In [35]:
import requests
import json
import os
import time
from sklearn.metrics import f1_score, classification_report
from collections import Counter
import pandas as pd
import re

# API Configuration
API_TOKEN = os.environ.get("OPENWEBUI_API_KEY")
API_URL = os.environ["OPENWEBUI_URL"] + "/api/chat/completions"

headers = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}

def extract_emotion_label(response_text):
    """Extract emotion label from LLM response."""
    if not response_text:
        return "neutral"

    response_text = response_text.lower().strip()
    valid_emotions = ["happiness", "sadness", "anger", "surprise", "fear", "disgust", "neutral"]

    emotion_pattern = r'emotion:\s*(\w+)'
    match = re.search(emotion_pattern, response_text, re.IGNORECASE)
    if match:
        extracted = match.group(1).lower()
        if extracted in valid_emotions:
            return extracted

    first_line = response_text.split('\n')[0].strip()
    for emotion in valid_emotions:
        if emotion in first_line:
            return emotion

    for emotion in valid_emotions:
        if emotion in response_text:
            return emotion

    return "neutral"

def classify_emotion_self_consistency(sentence, num_samples=3, show_full_response=False):
    """
    Self-consistency: Run multiple inferences and take majority vote.
    """
    predictions = []

    for i in range(num_samples):
        data = {
            "model": "meta-llama/Llama-3.3-70B-Instruct",
            "messages": [
                {
                    "role": "system",
                    "content": """You are an expert emotion classifier for Bulgarian text.

Classify into EXACTLY ONE emotion:
happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL INSTRUCTIONS:
- Only use "neutral" when there is truly NO emotional content
- If ANY emotion is present, classify it (not neutral)
- Prefer specific emotions over neutral when in doubt"""
                },
                {
                    "role": "user",
                    "content": f"""Examples:

HAPPINESS: Толкова съм щастлив днес!
SADNESS: Това ме разстройва много.
ANGER: Ядосан съм!
SURPRISE: Wow, не очаквах това!
FEAR: Страх ме е.
DISGUST: Това е отвратително.
NEUTRAL: Добре.

Input: {sentence}
Emotion:"""
                }
            ],
            "temperature": 0.3,
            "max_tokens": 100
        }

        try:
            response = requests.post(API_URL, headers=headers, json=data, timeout=90)
            if response.status_code == 200:
                result = response.json()
                full_response = result['choices'][0]['message']['content']
                extracted_label = extract_emotion_label(full_response)
                predictions.append(extracted_label)
            else:
                predictions.append("neutral")
        except Exception as e:
            predictions.append("neutral")

        time.sleep(0.3)  # Small delay between votes

    # Majority vote
    vote_counts = Counter(predictions)
    majority_emotion = vote_counts.most_common(1)[0][0]

    if show_full_response:
        print(f"  Votes: {predictions} → {majority_emotion}")

    return majority_emotion, str(predictions)

# Load full dataset
df = pd.read_csv("Test_set.csv")
df = df.rename(columns={"Sentence": "Sentence", "Emotion_core": "Label"})

print("=" * 60)
print("SELF-CONSISTENCY - FULL DATASET")
print("=" * 60)
print(f"Total dataset size: {len(df)}")
print("Running 3 inferences per sample with majority vote")
print("Estimated time: 50-60 minutes")

sample_size = len(df)
sample_df = df.copy()

predictions_sc_full = []
votes_full = []

start_time = time.time()

for idx, row in sample_df.iterrows():
    sentence = row['Sentence']

    extracted, vote_str = classify_emotion_self_consistency(sentence, num_samples=3, show_full_response=False)

    predictions_sc_full.append(extracted)
    votes_full.append(vote_str)

    if (idx + 1) % 50 == 0:
        elapsed = (time.time() - start_time) / 60
        avg_time = elapsed / (idx + 1)
        remaining = avg_time * (sample_size - (idx + 1))
        print(f"Progress: {idx+1}/{sample_size} | Elapsed: {elapsed:.1f}min | Est. remaining: {remaining:.1f}min")

    time.sleep(1)

sample_df['predicted_sc'] = predictions_sc_full
sample_df['votes'] = votes_full

# Evaluation
y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted_sc'].str.lower()

f1_macro_final = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted_final = f1_score(y_true, y_pred, average='weighted', zero_division=0)
matches = (y_true == y_pred).sum()
accuracy_final = matches / len(sample_df)

print("\n" + "=" * 60)
print("FINAL RESULTS - SELF-CONSISTENCY FULL DATASET")
print("=" * 60)
print(f"Sample Size: {sample_size}")
print(f"Correct Predictions: {matches}/{sample_size}")
print(f"Accuracy: {accuracy_final:.4f}")

SELF-CONSISTENCY - FULL DATASET
Total dataset size: 1005
Running 3 inferences per sample with majority vote
Estimated time: 50-60 minutes
Progress: 50/1005 | Elapsed: 3.2min | Est. remaining: 61.1min
Progress: 100/1005 | Elapsed: 6.0min | Est. remaining: 54.4min
Progress: 150/1005 | Elapsed: 9.2min | Est. remaining: 52.4min
Progress: 200/1005 | Elapsed: 12.1min | Est. remaining: 48.6min
Progress: 250/1005 | Elapsed: 15.0min | Est. remaining: 45.2min
Progress: 300/1005 | Elapsed: 17.6min | Est. remaining: 41.3min
Progress: 350/1005 | Elapsed: 20.6min | Est. remaining: 38.6min
Progress: 400/1005 | Elapsed: 23.5min | Est. remaining: 35.6min
Progress: 450/1005 | Elapsed: 26.3min | Est. remaining: 32.5min
Progress: 500/1005 | Elapsed: 28.8min | Est. remaining: 29.1min
Progress: 550/1005 | Elapsed: 32.0min | Est. remaining: 26.5min
Progress: 600/1005 | Elapsed: 34.8min | Est. remaining: 23.5min
Progress: 650/1005 | Elapsed: 38.1min | Est. remaining: 20.8min
Progress: 700/1005 | Elapsed: 40.8

In [36]:
# Evaluation
y_true = sample_df['Label'].str.lower()
y_pred = sample_df['predicted_sc'].str.lower()
f1_macro_final = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted_final = f1_score(y_true, y_pred, average='weighted', zero_division=0)
matches = (y_true == y_pred).sum()
accuracy_final = matches / len(sample_df)
print(f"F1 Score (Macro): {f1_macro_final:.4f}")
print(f"F1 Score (Weighted): {f1_weighted_final:.4f}")
print(f"Target F1 Weighted: 0.85")
print(f"Gap to Target: {0.85 - f1_weighted_final:.4f}")
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, zero_division=0))

F1 Score (Macro): 0.5301
F1 Score (Weighted): 0.6701
Target F1 Weighted: 0.85
Gap to Target: 0.1799

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger       0.32      0.47      0.38        51
     disgust       0.50      0.36      0.42        22
        fear       0.82      0.35      0.49        66
   happiness       0.87      0.55      0.68       187
     neutral       0.70      0.85      0.77       452
     sadness       0.74      0.61      0.67       188
    surprise       0.24      0.41      0.30        39

    accuracy                           0.67      1005
   macro avg       0.60      0.51      0.53      1005
weighted avg       0.71      0.67      0.67      1005



## Iteration 17: Majority vote with 5 votes and more examples for higher reliability

In [19]:
def classify_emotion_self_consistency_5votes(sentence, num_samples=5, show_full_response=False):
    """
    Self-consistency with 5 votes instead of 3 for higher reliability
    This is your working prompt - just more votes.
    """
    predictions = []

    for i in range(num_samples):
        data = {
            "model": "meta-llama/Llama-3.3-70B-Instruct",
            "messages": [
                {
                    "role": "system",
                    "content": """You are an expert emotion classifier for Bulgarian text.

Classify into EXACTLY ONE emotion:
happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL INSTRUCTIONS:
- Only use "neutral" when there is truly NO emotional content
- If ANY emotion is present, classify it (not neutral)
- Prefer specific emotions over neutral when in doubt"""
                },
                {
                    "role": "user",
                    "content": f"""Examples:

HAPPINESS: Толкова съм щастлив днес!
SADNESS: Това ме разстройва много.
ANGER: Ядосан съм!
SURPRISE: Wow, не очаквах това!
FEAR: Страх ме е.
DISGUST: Това е отвратително.
NEUTRAL: Добре.

Input: {sentence}
Emotion:"""
                }
            ],
            "temperature": 0.3,
            "max_tokens": 50
        }

        try:
            response = requests.post(API_URL, headers=headers, json=data, timeout=90)
            if response.status_code == 200:
                result = response.json()
                full_response = result['choices'][0]['message']['content']
                extracted_label = extract_emotion_label(full_response)
                predictions.append(extracted_label)
            else:
                predictions.append("neutral")
        except Exception as e:
            predictions.append("neutral")

        time.sleep(0.3)

    from collections import Counter
    vote_counts = Counter(predictions)
    majority_emotion = vote_counts.most_common(1)[0][0]
    vote_confidence = vote_counts[majority_emotion] / num_samples

    if show_full_response:
        print(f"  Votes: {predictions} → {majority_emotion} (confidence: {vote_confidence:.2f})")

    return majority_emotion, str(predictions)

# Test on 50 samples first to verify improvement
print("=" * 60)
print("SELF-CONSISTENCY WITH 5 VOTES (50 samples test)")
print("=" * 60)
print("Change: 3 votes → 5 votes for higher confidence")

test_size = 50
test_df = df.head(test_size).copy()

predictions_5votes = []

start_time = time.time()

for idx, row in test_df.iterrows():
    sentence = row['Sentence']
    extracted, votes = classify_emotion_self_consistency_5votes(sentence, num_samples=5, show_full_response=False)
    predictions_5votes.append(extracted)

    if (idx + 1) % 10 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"Progress: {idx+1}/{test_size} | Elapsed: {elapsed:.1f}min")

    time.sleep(1)

test_df['predicted_5votes'] = predictions_5votes

y_true = test_df['Label'].str.lower()
y_pred = test_df['predicted_5votes'].str.lower()

f1_weighted_5votes = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print("\n" + "=" * 60)
print("RESULTS COMPARISON (50 samples)")
print("=" * 60)
print(f"3 votes: F1 Weighted ~0.70")
print(f"5 votes: F1 Weighted {f1_weighted_5votes:.4f}")
print(f"Change: {f1_weighted_5votes - 0.70:+.4f}")

print("\n" + classification_report(y_true, y_pred, zero_division=0))

test_df.to_csv("self_consistency_5votes_test.csv", index=False)

# If improvement shown, run on full dataset
if f1_weighted_5votes > 0.70:
    print("\n" + "=" * 60)
    print("IMPROVEMENT DETECTED - PROCEEDING WITH FULL DATASET")
    print("=" * 60)

    user_confirm = input("Run on full 1005 samples? This will take ~45-60 minutes (y/n): ")

    if user_confirm.lower() == 'y':
        sample_df = df.copy()
        predictions_full = []

        start_time_full = time.time()

        for idx, row in sample_df.iterrows():
            sentence = row['Sentence']
            extracted, votes = classify_emotion_self_consistency_5votes(sentence, num_samples=5, show_full_response=False)
            predictions_full.append(extracted)

            if (idx + 1) % 50 == 0:
                elapsed = (time.time() - start_time_full) / 60
                avg_time = elapsed / (idx + 1)
                remaining = avg_time * (len(sample_df) - (idx + 1))
                print(f"Progress: {idx+1}/{len(sample_df)} | Elapsed: {elapsed:.1f}min | Est. remaining: {remaining:.1f}min")

            time.sleep(1)

        sample_df['predicted_5votes'] = predictions_full

        y_true_full = sample_df['Label'].str.lower()
        y_pred_full = sample_df['predicted_5votes'].str.lower()

        f1_weighted_full = f1_score(y_true_full, y_pred_full, average='weighted', zero_division=0)

        print("\n" + "=" * 60)
        print("FINAL RESULTS - 5 VOTES ON FULL DATASET")
        print("=" * 60)
        print(f"F1 Weighted: {f1_weighted_full:.4f}")
        print(f"Target: 0.85")
        print(f"Gap: {0.85 - f1_weighted_full:.4f}")

        if f1_weighted_full >= 0.85:
            print("\n*** TARGET ACHIEVED ***")

        print("\n" + classification_report(y_true_full, y_pred_full, zero_division=0))

        sample_df.to_csv("final_5votes_full_dataset.csv", index=False)
        print("\nResults saved to final_5votes_full_dataset.csv")
else:
    print("\nNo improvement from 5 votes. Stick with 3-vote version (F1 0.6981)")

SELF-CONSISTENCY WITH 5 VOTES (50 samples test)
Change: 3 votes → 5 votes for higher confidence
Progress: 10/50 | Elapsed: 1.0min
Progress: 20/50 | Elapsed: 1.7min
Progress: 30/50 | Elapsed: 2.5min
Progress: 40/50 | Elapsed: 3.6min
Progress: 50/50 | Elapsed: 4.3min

RESULTS COMPARISON (50 samples)
3 votes: F1 Weighted ~0.70
5 votes: F1 Weighted 0.6873
Change: -0.0127

              precision    recall  f1-score   support

       anger       0.67      0.67      0.67         3
     disgust       0.00      0.00      0.00         1
        fear       1.00      0.62      0.77         8
   happiness       0.90      0.75      0.82        12
     neutral       0.54      0.70      0.61        10
     sadness       0.73      0.73      0.73        11
    surprise       0.38      0.60      0.46         5

    accuracy                           0.68        50
   macro avg       0.60      0.58      0.58        50
weighted avg       0.72      0.68      0.69        50


No improvement from 5 votes. St

## Iteration 18: Best-of-N Sampling with Anti-Neutral Bias Prompt on 70B

In [20]:
def classify_emotion_best_of_n(sentence, n=5, show_full_response=False):
    """
    Best-of-N sampling: Generate N responses and select the best one.

    Selection criteria:
    1. Non-neutral prediction preferred (to combat neutral bias)
    2. Shortest response (more confident/direct)
    3. First valid emotion found
    """
    predictions = []
    full_responses = []

    for i in range(n):
        data = {
            "model": "meta-llama/Llama-3.3-70B-Instruct",
            "messages": [
                {
                    "role": "system",
                    "content": """You are an expert emotion classifier for Bulgarian text.

Classify into EXACTLY ONE emotion:
happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL INSTRUCTIONS:
- Only use "neutral" when there is truly NO emotional content
- If ANY emotion is present, classify it (not neutral)
- Prefer specific emotions over neutral when in doubt"""
                },
                {
                    "role": "user",
                    "content": f"""Examples:

HAPPINESS: Толкова съм щастлив днес!
SADNESS: Това ме разстройва много.
ANGER: Ядосан съм!
SURPRISE: Wow, не очаквах това!
FEAR: Страх ме е.
DISGUST: Това е отвратително.
NEUTRAL: Добре.

Input: {sentence}
Emotion:"""
                }
            ],
            "temperature": 0.4,
            "max_tokens": 100
        }

        try:
            response = requests.post(API_URL, headers=headers, json=data, timeout=90)
            if response.status_code == 200:
                result = response.json()
                full_response = result['choices'][0]['message']['content']
                extracted_label = extract_emotion_label(full_response)
                predictions.append(extracted_label)
                full_responses.append(full_response)
            else:
                predictions.append("neutral")
                full_responses.append("error")
        except Exception as e:
            predictions.append("neutral")
            full_responses.append("error")

        time.sleep(0.3)

    # Best-of-N selection logic
    # Criteria 1: Prefer non-neutral predictions
    non_neutral_preds = [(pred, resp) for pred, resp in zip(predictions, full_responses) if pred != "neutral"]

    if non_neutral_preds:
        # Criteria 2: Among non-neutral, choose shortest response (most confident)
        best_pred, best_resp = min(non_neutral_preds, key=lambda x: len(x[1]))
    else:
        # All predictions were neutral, pick shortest
        best_pred, best_resp = min(zip(predictions, full_responses), key=lambda x: len(x[1]))

    if show_full_response:
        print(f"  All predictions: {predictions}")
        print(f"  Selected best: {best_pred}")

    return best_pred, str(predictions)

# Test Best-of-N on 50 samples
print("=" * 60)
print("BEST-OF-N SAMPLING (N=5, 50 samples test)")
print("=" * 60)
print("Selection criteria:")
print("1. Prefer non-neutral predictions")
print("2. Choose shortest/most confident response")
print("3. Higher temperature (0.4) for diversity")
print("=" * 60)

test_size = 50
test_df_bon = df.head(test_size).copy()

predictions_bon = []

start_time = time.time()

for idx, row in test_df_bon.iterrows():
    sentence = row['Sentence']
    extracted, all_preds = classify_emotion_best_of_n(sentence, n=5, show_full_response=False)
    predictions_bon.append(extracted)

    if (idx + 1) % 10 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"Progress: {idx+1}/{test_size} | Elapsed: {elapsed:.1f}min")

    time.sleep(1)

test_df_bon['predicted_bon'] = predictions_bon

y_true_bon = test_df_bon['Label'].str.lower()
y_pred_bon = test_df_bon['predicted_bon'].str.lower()

f1_weighted_bon = f1_score(y_true_bon, y_pred_bon, average='weighted', zero_division=0)

print("\n" + "=" * 60)
print("BEST-OF-N RESULTS (50 samples)")
print("=" * 60)
print(f"Self-consistency (3 votes): F1 Weighted ~0.70")
print(f"Best-of-N (N=5):            F1 Weighted {f1_weighted_bon:.4f}")
print(f"Change: {f1_weighted_bon - 0.70:+.4f}")

print("\n" + classification_report(y_true_bon, y_pred_bon, zero_division=0))

# Analyze neutral predictions
neutral_count_bon = (test_df_bon['predicted_bon'].str.lower() == 'neutral').sum()
print(f"\nNeutral predictions: {neutral_count_bon}/{test_size}")

test_df_bon.to_csv("best_of_n_test.csv", index=False)
print("\nResults saved to best_of_n_test.csv")

# If better, offer full run
if f1_weighted_bon > 0.71:
    print("\n" + "=" * 60)
    print("IMPROVEMENT DETECTED")
    print("=" * 60)
    print("Best-of-N shows better performance than self-consistency")
    print("Ready to run on full dataset if desired")
else:
    print("\nNo significant improvement. Self-consistency with 5 votes likely better option.")

BEST-OF-N SAMPLING (N=5, 50 samples test)
Selection criteria:
1. Prefer non-neutral predictions
2. Choose shortest/most confident response
3. Higher temperature (0.4) for diversity
Progress: 10/50 | Elapsed: 1.1min
Progress: 20/50 | Elapsed: 1.8min
Progress: 30/50 | Elapsed: 2.5min
Progress: 40/50 | Elapsed: 3.5min
Progress: 50/50 | Elapsed: 4.2min

BEST-OF-N RESULTS (50 samples)
Self-consistency (3 votes): F1 Weighted ~0.70
Best-of-N (N=5):            F1 Weighted 0.7129
Change: +0.0129

              precision    recall  f1-score   support

       anger       0.67      0.67      0.67         3
     disgust       1.00      1.00      1.00         1
        fear       1.00      0.62      0.77         8
   happiness       0.90      0.75      0.82        12
     neutral       0.58      0.70      0.64        10
     sadness       0.73      0.73      0.73        11
    surprise       0.38      0.60      0.46         5

    accuracy                           0.70        50
   macro avg       

## Iteration 19: Advanced Expert Prompt with Self-Consistency Voting

In [21]:
def classify_emotion_advanced_expert(sentence, num_samples=5, show_full_response=False):
    """
    Advanced expert prompt with self-consistency voting.
    """
    predictions = []

    for i in range(num_samples):
        data = {
            "model": "meta-llama/Llama-3.3-70B-Instruct",
            "messages": [
                {
                    "role": "system",
                    "content": """You are a seasoned computational linguist specializing in emotion classification for Bulgarian-language text.

Your expertise is required to develop an accurate and precise emotion classification model that assigns exactly one of the following emotions to any given Bulgarian sentence:

EMOTIONS: happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL CLASSIFICATION GUIDELINES:
- Assign only one emotion label per input text without overlap
- When any emotional nuance is detected, classify it as one of the specific emotions rather than neutral
- In cases of ambiguity, prioritize assigning specific emotions over neutral to reflect subtle emotional cues
- Provide clear, concise, and formal classifications consistent with expert-level standards for Bulgarian emotional semantics
- Leverage your advanced knowledge of Bulgarian linguistic nuances and emotion detection methodologies

EMOTION DEFINITIONS:

HAPPINESS - positive emotions, joy, satisfaction, contentment
Bulgarian indicators: щастлив, радвам се, страхотно, чудесно

SADNESS - sorrow, grief, melancholy, disappointment
Bulgarian indicators: тъжен, разстроен, жал ми е, мъка

ANGER - frustration, irritation, rage, active hostility
Bulgarian indicators: ядосан, бесен, раздразнен, сърдит

SURPRISE - astonishment, unexpectedness, shock
Bulgarian indicators: изненадан, шокиран, не очаквах, вау

FEAR - anxiety, worry, apprehension, dread
Bulgarian indicators: страх, уплашен, тревожа се, боя се

DISGUST - revulsion, aversion, repulsion
Bulgarian indicators: отвратително, гадно, противно, мерзко

NEUTRAL - factual statements with no emotional content
Bulgarian indicators: добре, разбирам, ясно (in unemotional context)

Respond with ONLY the emotion label in lowercase."""
                },
                {
                    "role": "user",
                    "content": f"""Apply your expert emotion classification to the following Bulgarian text.

REFERENCE EXAMPLES:

HAPPINESS:
- Толкова съм щастлив днес!
- Страхотно, благодаря!

SADNESS:
- Това ме разстройва много.
- Жал ми е за това.

ANGER:
- Ядосан съм на тази ситуация!
- Не мога да понасям това!

SURPRISE:
- Wow, не очаквах това!
- Не вярвам!

FEAR:
- Страх ме е от това.
- Много съм уплашен.

DISGUST:
- Това е отвратително.
- Гадно ми е.

NEUTRAL:
- Добре.
- Разбирам.

INPUT TEXT:
{sentence}

EMOTION:"""
                }
            ],
            "temperature": 0.3,
            "max_tokens": 50
        }

        try:
            response = requests.post(API_URL, headers=headers, json=data, timeout=90)
            if response.status_code == 200:
                result = response.json()
                full_response = result['choices'][0]['message']['content']

                if show_full_response:
                    print(f"  Full response: '{full_response}'")

                extracted_label = extract_emotion_label(full_response)
                predictions.append(extracted_label)
            else:
                predictions.append("neutral")
        except Exception as e:
            predictions.append("neutral")

        time.sleep(0.3)

    from collections import Counter
    vote_counts = Counter(predictions)
    majority_emotion = vote_counts.most_common(1)[0][0]

    if show_full_response:
        print(f"  Votes: {predictions} → {majority_emotion}")

    return majority_emotion, str(predictions)

# Test on 50 samples with full response visibility
print("=" * 60)
print("TESTING ADVANCED EXPERT PROMPT (FIXED)")
print("=" * 60)

test_size = 50
test_df = df.head(test_size).copy()

predictions_expert = []

# Test first sample with full response to verify
print("\nTesting first sample with full response:")
first_sentence = test_df.iloc[0]['Sentence']
first_true = test_df.iloc[0]['Label']
print(f"Sentence: {first_sentence}")
print(f"True label: {first_true}")
first_pred, first_votes = classify_emotion_advanced_expert(first_sentence, num_samples=3, show_full_response=True)
print(f"Prediction: {first_pred}")

print("\nIf this looks correct, continuing with full test...\n")

start_time = time.time()

for idx, row in test_df.iterrows():
    sentence = row['Sentence']
    extracted, votes = classify_emotion_advanced_expert(sentence, num_samples=5, show_full_response=False)
    predictions_expert.append(extracted)

    if (idx + 1) % 10 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"Progress: {idx+1}/{test_size} | Elapsed: {elapsed:.1f}min")

    time.sleep(1)

test_df['predicted_expert'] = predictions_expert

y_true = test_df['Label'].str.lower()
y_pred = test_df['predicted_expert'].str.lower()

f1_weighted_expert = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Simple 3-vote:     F1 Weighted 0.6981")
print(f"Advanced 5-vote:   F1 Weighted {f1_weighted_expert:.4f}")
print(f"Change: {f1_weighted_expert - 0.6981:+.4f}")

print("\n" + classification_report(y_true, y_pred, zero_division=0))

test_df.to_csv("advanced_expert_fixed_test.csv", index=False)

TESTING ADVANCED EXPERT PROMPT (FIXED)

Testing first sample with full response:
Sentence: Ако приемаш моите извинения, просто ме пригърни.
True label: sadness
  Full response: 'sadness'
  Full response: 'sadness'
  Full response: 'sadness'
  Votes: ['sadness', 'sadness', 'sadness'] → sadness
Prediction: sadness

If this looks correct, continuing with full test...

Progress: 10/50 | Elapsed: 0.6min
Progress: 20/50 | Elapsed: 1.2min
Progress: 30/50 | Elapsed: 1.8min
Progress: 40/50 | Elapsed: 2.4min
Progress: 50/50 | Elapsed: 3.0min

RESULTS
Simple 3-vote:     F1 Weighted 0.6981
Advanced 5-vote:   F1 Weighted 0.6668
Change: -0.0313

              precision    recall  f1-score   support

       anger       0.50      0.33      0.40         3
     disgust       0.00      0.00      0.00         1
        fear       1.00      0.75      0.86         8
   happiness       1.00      0.58      0.74        12
     neutral       0.42      1.00      0.59        10
     sadness       0.78      0.64  

In [22]:
# Diagnostic test - see EXACTLY what the model returns
test_sentence = "Толкова съм щастлив днес!"  # Clearly happiness

print("=" * 60)
print("DIAGNOSTIC TEST - RAW MODEL RESPONSE")
print("=" * 60)

data = {
    "model": "meta-llama/Llama-3.3-70B-Instruct",
    "messages": [
        {
            "role": "system",
            "content": """You are a seasoned computational linguist specializing in emotion classification for Bulgarian-language text.

Your expertise is required to develop an accurate and precise emotion classification model that assigns exactly one of the following emotions to any given Bulgarian sentence:

EMOTIONS: happiness, sadness, anger, surprise, fear, disgust, neutral

Respond with ONLY the emotion label in lowercase."""
        },
        {
            "role": "user",
            "content": f"""Examples:

HAPPINESS: Толкова съм щастлив днес!
SADNESS: Това ме разстройва много.
ANGER: Ядосан съм!

Input: {test_sentence}
Emotion:"""
        }
    ],
    "temperature": 0.3,
    "max_tokens": 100
}

response = requests.post(API_URL, headers=headers, json=data, timeout=90)
result = response.json()

print("\nFULL JSON RESPONSE:")
print(json.dumps(result, indent=2, ensure_ascii=False))

print("\nEXTRACTED TEXT:")
full_text = result['choices'][0]['message']['content']
print(f"'{full_text}'")

print("\nAFTER EXTRACTION FUNCTION:")
extracted = extract_emotion_label(full_text)
print(f"'{extracted}'")

DIAGNOSTIC TEST - RAW MODEL RESPONSE

FULL JSON RESPONSE:
{
  "id": "chatcmpl-6429224374224d00944a024229214334",
  "object": "chat.completion",
  "created": 1761579651,
  "model": "meta-llama/Llama-3.3-70B-Instruct",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "happiness",
        "refusal": null,
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [],
        "reasoning_content": null
      },
      "logprobs": null,
      "finish_reason": "stop",
      "stop_reason": null
    }
  ],
  "service_tier": null,
  "system_fingerprint": null,
  "usage": {
    "prompt_tokens": 230,
    "total_tokens": 233,
    "completion_tokens": 3,
    "prompt_tokens_details": null
  },
  "prompt_logprobs": null,
  "kv_transfer_params": null
}

EXTRACTED TEXT:
'happiness'

AFTER EXTRACTION FUNCTION:
'happiness'


## Iteration 20: Refined Self-Consistency with Enhanced Extraction

In [29]:
headers = {
    'Authorization': f'Bearer {API_TOKEN}',
    'Content-Type': 'application/json'
}

# Emotion Label Extraction Function
def extract_emotion_label(response_text):
    """
    Extracts the single emotion label from the LLM's response text.
    Handles variations like 'Emotion: label', 'label', etc.
    Defaults to 'neutral' if no valid label is found or on error.
    """
    if not response_text or not isinstance(response_text, str):
        # Return neutral if response is empty or not a string
        return "neutral"

    response_text = response_text.lower().strip()
    valid_emotions = ["happiness", "sadness", "anger", "surprise", "fear", "disgust", "neutral"]

    # Pattern 1: Look for "Emotion: label" explicitly or just the label at the start
    # Allows optional non-word chars before/after label
    emotion_pattern = r'(?:emotion:)?\s*(' + '|'.join(valid_emotions) + r')\b'
    match = re.match(emotion_pattern, response_text) # Use match to check start
    if match:
        extracted = match.group(1).lower()
        if extracted in valid_emotions:
            # Check if it's essentially the *only* thing in the response
            if re.fullmatch(r'\W*' + re.escape(extracted) + r'\W*', response_text):
                return extracted
            # If it starts with "Emotion:", also valid
            if response_text.startswith("emotion:"):
                 return extracted

    # Pattern 2: Check if the *entire* response is just one of the labels (allowing punctuation)
    for emotion in valid_emotions:
        if re.fullmatch(r'\W*' + re.escape(emotion) + r'\W*', response_text):
            return emotion

    # Pattern 3: Fallback - Find the first valid emotion mentioned anywhere in the response. Less reliable.
    for emotion in valid_emotions:
         # Use word boundaries to avoid matching parts of words
        if re.search(r'\b' + re.escape(emotion) + r'\b', response_text):
            print(f"  Warning: Using fallback extraction for '{response_text}', found '{emotion}'.")
            return emotion

    # Default if no specific emotion is clearly identified
    print(f"  Warning: Could not extract valid label from '{response_text}'. Defaulting to neutral.")
    return "neutral"


# Refined Self-Consistency Classification Function
def classify_emotion_refined_sc(sentence, num_samples=3, show_full_response=False):
    """
    Combines 70B model, balanced few-shot, anti-neutral bias,
    and self-consistency (3 votes), with increased max_tokens and explicit output format instruction.
    """
    predictions = []
    raw_responses = [] # Store raw responses for debugging if needed

    # Check if headers are valid before proceeding
    if not headers or 'Authorization' not in headers or headers['Authorization'] == 'Bearer YOUR_API_KEY_HERE':
         print("  Error: Invalid or missing API credentials in headers.")
         return "neutral", "['API_error']"

    for i in range(num_samples):
        data = {
            "model": "meta-llama/Llama-3.3-70B-Instruct",
            "messages": [
                {
                    "role": "system",
                    "content": """You are an expert emotion classifier for Bulgarian text.

Classify into EXACTLY ONE emotion:
happiness, sadness, anger, surprise, fear, disgust, neutral

CRITICAL INSTRUCTIONS:
- Only use "neutral" when there is truly NO emotional content.
- If ANY emotion is present, classify it (not neutral).
- Prefer specific emotions over neutral when in doubt.
- Neutral is for factual statements only, not mild emotions.
- Respond with ONLY the single emotion label in lowercase. Do not add explanations or introductions."""
                },
                {
                    "role": "user",
                    "content": f"""Classify the emotion in the Bulgarian text using these examples:

HAPPINESS examples:
- Толкова съм щастлив днес! → happiness
- Страхотно! → happiness
- Радвам се много! → happiness

SADNESS examples:
- Това ме разстройва много. → sadness
- Тъжен съм. → sadness
- Жал ми е. → sadness

ANGER examples:
- Ядосан съм на тази ситуация! → anger
- Не мога да понасям това! → anger
- Бесен съм! → anger

SURPRISE examples:
- Wow, не очаквах това! → surprise
- Не вярвам! → surprise
- Шокиран съм! → surprise

FEAR examples:
- Страх ме е от това. → fear
- Уплашен съм. → fear
- Тревожа се. → fear

DISGUST examples:
- Това е отвратително. → disgust
- Гадно ми е. → disgust
- Противно! → disgust

NEUTRAL examples:
- Добре. → neutral
- Разбирам. → neutral
- Ясно. → neutral

Now classify the following sentence:
Input: {sentence}
Emotion:"""
                }
            ],
            "temperature": 0.3,
            "max_tokens": 50
        }

        try:
            # Increased timeout for robustness
            response = requests.post(API_URL, headers=headers, json=data, timeout=120)

            if response.status_code == 200:
                result = response.json()
                if result and 'choices' in result and len(result['choices']) > 0:
                    full_response = result['choices'][0].get('message', {}).get('content', '')
                    raw_responses.append(full_response) # Store raw response
                    extracted_label = extract_emotion_label(full_response)
                    predictions.append(extracted_label)
                else:
                    print(f"  Warning: Unexpected API response format for sample {i+1}: {result}")
                    predictions.append("neutral")
                    raw_responses.append(f"ERROR: Unexpected format - {result}")
            else:
                # Log non-200 status codes with response text for debugging
                error_text = response.text[:500] # Limit error text length
                print(f"  Warning: API call {i+1} failed with status {response.status_code}. Response: {error_text}")
                predictions.append("neutral")
                raw_responses.append(f"ERROR: Status {response.status_code}")

        except requests.exceptions.Timeout:
            print(f"  Error: Request {i+1} timed out after 120 seconds.")
            predictions.append("neutral")
            raw_responses.append("ERROR: Timeout")
        except requests.exceptions.RequestException as e:
            # Catch other potential network/request errors
            print(f"  Error: Request {i+1} failed due to a connection/request issue: {e}")
            predictions.append("neutral")
            raw_responses.append(f"ERROR: RequestException - {e}")
        except Exception as e:
            # Catch any other unexpected errors during the API call or response processing
            print(f"  Error: An unexpected error occurred during API call {i+1}: {e}")
            predictions.append("neutral")
            raw_responses.append(f"ERROR: Exception - {e}")

        # Short delay between votes can sometimes help API stability or rate limits
        time.sleep(0.3)

    # --- Majority Vote Logic ---
    if not predictions: # Handle case where all API calls failed for a sentence
        final_emotion = "neutral"
        vote_str = "[]"
        print("  Warning: No predictions generated for this sentence, defaulting to neutral.")
    else:
        vote_counts = Counter(predictions)
        most_common = vote_counts.most_common()

        # Tie-breaking: prefer non-neutral if counts are equal
        if len(most_common) > 1 and most_common[0][1] == most_common[1][1]:
            non_neutral_options = [item[0] for item in most_common if item[0] != "neutral"]
            if non_neutral_options:
                # Pick the alphabetically first non-neutral option for deterministic tie-breaking
                final_emotion = sorted(non_neutral_options)[0]
            else:
                final_emotion = "neutral" # Only neutral options were tied
        elif len(most_common) >= 1:
             final_emotion = most_common[0][0]
        else: # Should not happen if predictions list is not empty
             final_emotion = "neutral"
             print("  Error: Logical inconsistency in majority vote.")

        vote_str = str(predictions) # Store the list of votes as a string representation

    # Optional: Log the voting process if needed for debugging
    if show_full_response:
        print(f"  Raw Responses: {raw_responses}")
        print(f"  Votes: {predictions} -> {final_emotion}")

    return final_emotion, vote_str

# Run Classification on Full Dataset:
if 'df' in locals() and isinstance(df, pd.DataFrame) and "Sentence" in df.columns and "Label" in df.columns:
    print("\n" + "=" * 60)
    print("STARTING CLASSIFICATION ON PREVIOUSLY LOADED DATAFRAME ('df')")
    print("Using: 70B Model, Refined SC Prompt (3 Votes)")
    print(f"DataFrame size: {len(df)} rows")
    print("=" * 60)

    sample_size = len(df)
    # Create a copy to store results without modifying the original df immediately
    # If you want to add columns directly to the existing 'df', remove .copy()
    results_df = df.copy()

    predictions_final = []
    votes_final = []

    start_time = time.time()

    for idx, row in results_df.iterrows():
        # Check for valid sentence data before processing
        sentence = row.get('Sentence') # Use .get to avoid KeyError if column somehow missing
        if pd.isna(sentence) or not isinstance(sentence, str) or not sentence.strip():
            print(f"Skipping row {idx+1}: Invalid or empty sentence data.")
            predictions_final.append("neutral") # Default prediction for bad data
            votes_final.append("['skipped']")
            continue # Move to the next row

        sentence = sentence.strip()

        # Get prediction and votes using the refined function
        extracted_emotion, vote_str = classify_emotion_refined_sc(sentence, num_samples=3, show_full_response=False)

        predictions_final.append(extracted_emotion)
        votes_final.append(vote_str)

        # Progress Update Logic
        current_item = idx + 1
        if current_item % 50 == 0 or current_item == sample_size:
            elapsed_min = (time.time() - start_time) / 60
            if current_item > 0:
                avg_time_per_item_min = elapsed_min / current_item
                remaining_items = sample_size - current_item
                remaining_time_min = avg_time_per_item_min * remaining_items
                print(f"Progress: {current_item}/{sample_size} | Elapsed: {elapsed_min:.1f}min | Est. remaining: {remaining_time_min:.1f}min")
            else:
                # Avoid division by zero if somehow the first item fails instantly
                 print(f"Progress: {current_item}/{sample_size} | Elapsed: {elapsed_min:.1f}min")

        # Rate limiting - pause between processing each row
        time.sleep(0.5) # Adjust if API has stricter rate limits

    # Add results to the DataFrame after the loop finishes
    # If using df.copy(), this updates the copy. If not using copy(), it updates the original df.
    results_df['predicted_emotion'] = predictions_final
    results_df['votes'] = votes_final

    total_time_min = (time.time() - start_time) / 60
    print(f"\nClassification finished in {total_time_min:.2f} minutes.")

    # %% Evaluate Results
    print("\n" + "=" * 60)
    print("EVALUATION")
    print("=" * 60)

    # Ensure labels are strings and handle potential NaN in ground truth
    y_true = results_df['Label'].fillna('neutral').astype(str).str.lower()
    y_pred = results_df['predicted_emotion'].fillna('neutral').astype(str).str.lower()

    # Calculate Metrics
    accuracy = accuracy_score(y_true, y_pred)
    # Use macro average for F1 score as requested in analysis
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    # Also calculate weighted average for overall performance context
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"Sample Size: {sample_size}")
    print(f"Correct Predictions: {int(accuracy * sample_size)}/{sample_size}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score (Macro): {f1_macro:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")
    print("-" * 30)
    # Task requirement check based on Weighted F1
    target_f1 = 0.85
    print(f"Target F1 Weighted: > {target_f1}")

    if f1_weighted > target_f1:
        print("TARGET ACHIEVED!")
        gap = f1_weighted - target_f1
        print(f"Exceeded target by: +{gap:.4f}")
    else:
        print("Target not met.")
        gap = target_f1 - f1_weighted
        print(f"Gap to target: {gap:.4f}")

    # Classification Report
    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT")
    print("=" * 60)
    # Ensure all possible labels are included for a complete report
    all_labels = sorted(list(set(y_true) | set(y_pred)))
    print(classification_report(y_true, y_pred, labels=all_labels, zero_division=0))

    # %% Save Results
    output_csv = "refined_sc_results_full.csv" # Define output file name here
    print("\n" + "=" * 60)
    print(f"SAVING RESULTS TO {output_csv}")
    print("=" * 60)
    try:
        # Save the DataFrame containing the results
        results_df.to_csv(output_csv, index=False, encoding='utf-8')
        print(f"Successfully saved results to {output_csv}")

    except Exception as e:
        print(f"Error saving results to CSV: {e}")

else:
    print("\nDataFrame 'df' not found or is invalid. Skipping classification and evaluation.")
    print("Please ensure 'df' is loaded correctly in a previous cell.")


STARTING CLASSIFICATION ON PREVIOUSLY LOADED DATAFRAME ('df')
Using: 70B Model, Refined SC Prompt (3 Votes)
DataFrame size: 1005 rows
Progress: 50/1005 | Elapsed: 2.0min | Est. remaining: 37.4min
Progress: 100/1005 | Elapsed: 3.9min | Est. remaining: 34.9min
Progress: 150/1005 | Elapsed: 6.0min | Est. remaining: 34.1min
Progress: 200/1005 | Elapsed: 7.8min | Est. remaining: 31.4min
Progress: 250/1005 | Elapsed: 9.6min | Est. remaining: 29.0min
Progress: 300/1005 | Elapsed: 11.4min | Est. remaining: 26.9min
Progress: 350/1005 | Elapsed: 13.8min | Est. remaining: 25.8min
Progress: 400/1005 | Elapsed: 15.9min | Est. remaining: 24.0min
Progress: 450/1005 | Elapsed: 18.0min | Est. remaining: 22.2min
Progress: 500/1005 | Elapsed: 20.4min | Est. remaining: 20.6min
Progress: 550/1005 | Elapsed: 22.6min | Est. remaining: 18.7min
Progress: 600/1005 | Elapsed: 24.9min | Est. remaining: 16.8min

neutral', found 'neutral'.

neutral', found 'neutral'.

neutral', found 'happiness'.
Progress: 650/100